# CFEVER：三階段事實查核作業

本範例的三個階段都使用中文 BERT：

1. **Document Retrieval**：找出與 claim 相關的 Wikipedia 頁面。
2. **Evidence Retrieval**：從這些頁面挑選證據句子。
3. **Classification**：讓 BERT 根據 claim 和證據預測標籤。

所有程式碼已完整提供，請由上而下執行。
為了縮短課堂測試時間，語料只使用公開訓練集涉及的頁面與證據句。
驗證集用於觀察表現，測試集只用來產生提交結果。



## 1. 套件與參數

請先啟用 GPU。優先從 Kaggle Input 載入完整模型，便可離線執行。
第一次從 Hugging Face 下載時需要 Internet；下載後須把模型加入 Input，供正式重跑使用。



In [1]:
# Generative AI 使用聲明：本 notebook 的程式碼由 Anthropic Claude（Sonnet 5.5 / Opus 5.5）協助撰寫，
# 使用方式與使用的預訓練模型見最後一個 cell（Report）的 Q5。
import os, re, json, math, time, random, pickle, gc, functools, collections
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold


_SPACE = re.compile(r"(?<=[^\x00-\x7f])\s+|\s+(?=[^\x00-\x7f])")   # 只去掉與中文字（非 ASCII）相鄰的空白
_PAREN = re.compile(r"_\(.*\)$")


def norm(s):
    """wiki 文字是斷詞過的（『數學 ， 是研究…』），claim 沒有空格；統一去掉與中文相鄰的空白。"""
    return _SPACE.sub("", s).strip()


def title_base(t):
    """'愛爾蘭_(消歧義)' -> '愛爾蘭'（用來和 claim 做子字串比對）"""
    return _PAREN.sub("", t).replace("_", " ")


def bigrams(s):
    if len(s) < 2:
        return {s} if s else set()
    return {s[i:i + 2] for i in range(len(s) - 1)}


def read_jsonl(path):
    with open(path, encoding="utf-8-sig") as f:
        return [json.loads(l) for l in f if l.strip()]


def free_gpu():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def amp_ctx():
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)


def cached(path, build_fn, force=False):
    """有快取就讀，沒有就執行 build_fn 並存檔。"""
    path = Path(path)
    if path.exists() and not force:
        with open(path, "rb") as f:
            return pickle.load(f)
    obj = build_fn()
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    return obj


LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}
LABEL_OUT = ["supports", "refutes", "NOT ENOUGH INFO"]   # 與 sample code 的輸出格式一致


def parse_gold(row):
    """回傳 [[(page_id, sentence_id), ...], ...]；NEI 沒有證據，回傳 []。"""
    if row["label"].upper() == "NOT ENOUGH INFO":
        return []
    groups = []
    for g in row["evidence"]:
        items = [(it[2], int(it[3])) for it in g if it[2] is not None and it[3] is not None]
        if items:
            groups.append(items)
    return groups

/home/yiting/venvs/hw3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------- 最重要的開關 ----------
QUICK_TEST = False        # True：用 wiki_sample.jsonl 與少量資料，十分鐘內跑完整條流程（先用它檢查環境）
FORCE_RETRAIN = False     # True：忽略已存的模型快取，重新訓練（包含重排器，約 40 分鐘）
FORCE_RETRAIN_CLASSIFIER = False   # True：只重新訓練分類器
SEED = 2026

# ---------- 檢索參數 ----------
LEAD_RAW_CHARS, LEAD_CHARS = 400, 110   # 頁面「摘要」= 標題 + 前 110 字（BM25 與稠密檢索使用）
BM25_FEATURES, BM25_K1, BM25_B, BM25_MAX_DF = 2 ** 21, 1.2, 0.75, 0.05
K_BM25, K_DENSE = 50, 50                # BM25 / 稠密檢索各取前幾名
CAND_BM25, CAND_DENSE = 20, 20          # 其中前幾名進入候選集
TITLE_MIN, TITLE_MAX = 2, 24            # 標題比對的長度範圍（單字標題噪音太大，直接略過）
PAGES_PER_TITLE, MAX_TITLE_CANDS = 6, 40
USE_DENSE = True
DENSE_MODEL = "BAAI/bge-small-zh-v1.5"
DENSE_MAX_LEN, DENSE_BS = 96, 256

# ---------- 句子檢索 ----------
PAGE_TOPK = 6                           # 取排序後前幾頁的句子當候選
PER_PAGE_MAX_SID = 60                   # 每頁只看前 60 句（gold 有 95% 在 50 句內）
RERANK_POOL = 32                        # 每個 claim 送進 cross-encoder 的句子數
USE_HOP2 = True                         # 是否沿超連結做第二跳
HOP_SRC, HOP_MAX_PAGES, HOP_PER_PAGE, HOP_POOL = 2, 6, 4, 16
RERANK_MODEL = "hfl/chinese-macbert-base"
RERANK_MAX_LEN, RERANK_NEG = 128, 5
RERANK_EPOCHS, RERANK_LR, RERANK_BS, RERANK_ACCUM = 1, 3e-5, 16, 2
RERANK_TRAIN_CLAIMS = None              # None = 使用全部 supports/refutes 訓練題；顯存/時間不夠可設成 8000

# ---------- 分類器 ----------
CLS_MODEL = "hfl/chinese-macbert-base"  # 若載入失敗可改成 "google-bert/bert-base-chinese"
CLS_CTX_K, CLS_MAX_LEN = 4, 256
CLS_EPOCHS, CLS_LR, CLS_BS, CLS_ACCUM = 2, 2e-5, 8, 4
# 分類器集成（v2）：多個不同骨幹/種子的分類器，logits（log-softmax）取平均。每個成員訓練完都存檔，之後重跑會直接載入。
#   - macbert_base 是 v1 已訓練好的分類器（classifier_v2），不會重訓。
#   - 想加成員就把 tag 加進 ACTIVE_MEMBERS：先跑 roberta_base（約 35 分鐘），確認有進步再加 macbert_large（約 1.5~2 小時）。
CLS_MEMBERS = [
    dict(tag="macbert_base", model="hfl/chinese-macbert-base", path="classifier_v2", max_len=CLS_MAX_LEN),
    dict(tag="roberta_base", model="hfl/chinese-roberta-wwm-ext", lr=2e-5, epochs=2, bs=8, accum=4, seed=11),
    # 先在中文 NLI 資料（CMNLI/OCNLI 等，約 100 萬筆）微調過的 checkpoint（IDEA-CCNL Erlangshen，Apache-2.0，皆 < 1B）。
    # NLI 的 (前提, 假設) 對應這裡的 (證據, claim)，所以 swap=True 把證據放前面；NLI 標籤順序是 矛盾/中立/蘊含，
    # head_perm 把分類頭重排成 supports/refutes/NEI = 蘊含/矛盾/中立，讓微調從「已經懂得推論」的起點開始。
    dict(tag="erlangshen_110m", model="IDEA-CCNL/Erlangshen-Roberta-110M-NLI", lr=2e-5, epochs=2, bs=8, accum=4, seed=21,
         swap=True, head_perm=[2, 0, 1]),
    # 330M（large）：4GB 顯存放不下完整 AdamW，改用 Adafactor + 凍結 embedding 與前 8 層（OOM 時自動多凍結 4 層再重試）。
    # 注意：不要用 Erlangshen-MegatronBert-1.3B-NLI，它超過助教規定的 1B 上限。
    dict(tag="erlangshen_330m", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", lr=1.5e-5, epochs=2, bs=4, accum=8, seed=22,
         swap=True, head_perm=[2, 0, 1], freeze=8, optim="adafactor", infer_bs=16),
    # 同一個 330M 架構的第二顆種子（約 2 小時）。seed 會同時改變分類頭初始化、dropout、資料順序與訓練資料的建立順序，
    # 兩顆模型的錯誤因此不完全重疊，平均後變異數下降 —— 這是「真的」泛化提升（可在 dev 驗證），不是照 public 挑出來的。
    dict(tag="erlangshen_330m_b", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", lr=1.5e-5, epochs=2, bs=4, accum=8, seed=42,
         swap=True, head_perm=[2, 0, 1], freeze=8, optim="adafactor", infer_bs=16),
    # 小型 LLM（Qwen3-0.6B-Base，約 0.6B 參數 < 1B）+ LoRA。需要 peft（pip install peft）。
    # 先跑 qwen06_pilot（只用 6000 筆訓練資料、1 epoch）：dev accuracy 若已達 ~0.72 以上就值得跑完整版 qwen06。
    dict(tag="qwen06_pilot", kind="llm", model="Qwen/Qwen3-0.6B-Base", lr=1e-4, epochs=1, bs=4, accum=4, seed=31, n_train=6000,
         max_len=384, infer_bs=16),
    dict(tag="qwen06", kind="llm", model="Qwen/Qwen3-0.6B-Base", lr=1e-4, epochs=2, bs=4, accum=4, seed=32, max_len=384, infer_bs=16),   # v9 使用（見第 9 節）
]
ACTIVE_MEMBERS = ["macbert_base", "roberta_base", "erlangshen_110m", "erlangshen_330m", "erlangshen_330m_b", "qwen06_pilot"]
NEW_MEMBER = "erlangshen_330m_b"        # 下面會做「有/沒有這個成員」的配對比較，判斷提升是不是雜訊
PRIOR_TARGET = (1 / 3, 1 / 3, 1 / 3)    # 僅用於「舊的先驗修正」消融；正式預測改用 dev 校準（見分類評估 cell）
CLS_TRAIN_SR_CLAIMS = None              # None = 所有 supports/refutes 訓練題都用重排器重排一次，當作分類訓練證據（約 25 分鐘）
CLS_SHUFFLE_P = 0.3                     # 訓練時有 30% 的機率把證據順序打亂，避免模型依賴「標準證據排第一」
HOP_PENALTIES = (0.0, 1.0, 2.0, 3.0, 5.0, None)   # 第二跳候選的扣分；None = 不使用第二跳。依 dev 選擇

# ---------- 共用 ----------
GRAD_CKPT = True                        # gradient checkpointing：省顯存（約慢 30%）。顯存夠可設 False
INFER_BS = 64
N_TRAIN_LIMIT = N_DEV_LIMIT = N_TEST_LIMIT = None

if QUICK_TEST:
    N_TRAIN_LIMIT, N_DEV_LIMIT, N_TEST_LIMIT = 800, 200, 100
    RERANK_TRAIN_CLAIMS = 400
    RERANK_EPOCHS = CLS_EPOCHS = 1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("裝置：", device, torch.cuda.get_device_name(0) if USE_AMP else "（沒有 GPU，會非常慢）")

# ---------- 路徑 ----------
IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    DATA_DIR, OUTPUT_ROOT = Path("/kaggle/input"), Path("/kaggle/working")
else:
    DATA_DIR = next((p for p in (Path("dataset"), Path(".")) if (p / "train.jsonl").is_file()), Path("dataset"))
    OUTPUT_ROOT = Path("output")
VERSION = "final"                                  # 提交檔與 metrics.json 放進 output/<VERSION>/，每個版本各一個資料夾
OUTPUT_DIR = OUTPUT_ROOT / VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_file(name):
    direct = DATA_DIR / name
    if direct.is_file():
        return direct
    hits = sorted(p for p in DATA_DIR.rglob(name) if p.is_file())
    if not hits:
        raise FileNotFoundError(f"找不到 {name}；目前搜尋目錄：{DATA_DIR.resolve()}")
    return hits[0]


TRAIN_PATH, DEV_PATH, TEST_PATH = find_file("train.jsonl"), find_file("dev.jsonl"), find_file("test.jsonl")
WIKI_PATH = find_file("wiki_sample.jsonl" if QUICK_TEST else "wiki_pages.jsonl")
CACHE_DIR = OUTPUT_ROOT / ("cache_quick" if QUICK_TEST else "cache")   # 快取不跟著版本走（否則所有模型與推論都要重算）
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("資料夾：", DATA_DIR.resolve(), "| wiki：", WIKI_PATH.name, "| 輸出：", OUTPUT_DIR.resolve())

裝置： cuda NVIDIA GeForce RTX 3050 Laptop GPU
資料夾： /home/yiting/NLP-Assignment3/dataset | wiki： wiki_pages.jsonl | 輸出： /home/yiting/NLP-Assignment3/output/final


### 讀取資料

JSONL 每一行是一筆 JSON。`label` 是標準標籤，`evidence` 是標準證據群組。
`all_train_df` 保留全部公開訓練資料，供建立檢索語料使用。
`train_df` 是本次用來訓練 BERT 的抽樣資料。
使用開啟的檔案物件讀取，避免把路徑字串誤判為 JSON 內容。



In [3]:
# ----- 讀取 claim，建立 train / dev / test -----
def make_split(rows, name, labeled=True):
    s = dict(name=name, id=[r["id"] for r in rows], claim=[norm(r["claim"]) for r in rows])
    if labeled:
        s["label"] = np.array([LABEL2ID[r["label"].upper()] for r in rows])
        s["gold"] = [parse_gold(r) for r in rows]
    return s


rng = random.Random(SEED)
train_rows, dev_rows, test_rows = read_jsonl(TRAIN_PATH), read_jsonl(DEV_PATH), read_jsonl(TEST_PATH)
train = make_split(train_rows, "train")
dev = make_split(dev_rows, "dev")
test = make_split(test_rows, "test", labeled=False)
for s in (train, dev, test):
    print(s["name"], len(s["claim"]), (np.bincount(s["label"], minlength=3).tolist() if "label" in s else ""))

train 24012 [11085, 7113, 5814]
dev 3000 [1000, 1000, 1000]
test 6000 


### 下載中文 BERT

先尋找 Input 中的完整 BERT 模型，找不到才考慮下載至工作目錄。
`/kaggle/working` 的互動執行檔案不會自動變成下一次提交的 Input。
`.part` 是尚未完成的下載檔；完整下載後才改成正式檔名。
如需離線重跑，請將模型資料夾以 Dataset 或 Notebook Output 加入 Input。



In [4]:
# 載入 / 微調預訓練模型的共用函式（各階段都會用到；模型首次使用時由 Hugging Face 下載）


def load_dense_model():
    tok = AutoTokenizer.from_pretrained(DENSE_MODEL)
    model = AutoModel.from_pretrained(DENSE_MODEL).to(device).eval()
    if device.type == "cuda":
        model = model.half()
    return tok, model


# ----- 通用：微調序列分類模型 / 推論 -----
def _new_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_seq_clf(examples, num_labels, model_name, max_len, epochs, lr, micro_bs, accum, desc, seed=None, freeze=0, optim="adamw", head_perm=None):
    """examples: [(text_a, text_b, label), ...]。標準流程：AdamW + 線性 warmup/decay + fp16 + 梯度累積。"""
    seed = SEED if seed is None else seed
    torch.manual_seed(seed)                                   # 不同種子 → 分類頭初始化與 dropout 不同（集成需要多樣性）
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)
    if head_perm is not None and hasattr(model, "classifier") and model.classifier.out_features == len(head_perm):
        with torch.no_grad():                                 # 把 NLI 分類頭的類別順序換成我們的順序
            pm = torch.tensor(head_perm, device=model.classifier.weight.device)
            model.classifier.weight.copy_(model.classifier.weight[pm].clone())
            model.classifier.bias.copy_(model.classifier.bias[pm].clone())
    if freeze:                                                # 凍結 embedding 與前 freeze 層：省顯存、也省反向傳播時間
        base = model.base_model
        for p in base.embeddings.parameters():
            p.requires_grad = False
        for layer in base.encoder.layer[:freeze]:
            for p in layer.parameters():
                p.requires_grad = False
    if GRAD_CKPT:
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.gradient_checkpointing_enable()
    params = [p for p in model.parameters() if p.requires_grad]
    if optim == "adafactor":                                  # 優化器狀態極小（無動量、二階矩分解），large 模型才放得進 4GB
        from transformers.optimization import Adafactor
        opt = Adafactor(params, lr=lr, scale_parameter=False, relative_step=False, warmup_init=False, beta1=None, weight_decay=0.0)
    else:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    steps = math.ceil(len(examples) / (micro_bs * accum)) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = _new_scaler()
    rg = random.Random(seed)
    for ep in range(epochs):
        model.train()
        order = list(range(len(examples)))
        rg.shuffle(order)
        pbar = tqdm(range(0, len(order), micro_bs), desc=f"{desc} epoch {ep + 1}/{epochs}", mininterval=10)
        run_loss, pending = 0.0, 0
        for bi, s in enumerate(pbar):
            batch = [examples[i] for i in order[s:s + micro_bs]]
            enc = tok([b[0] for b in batch], [b[1] for b in batch], truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            labels = torch.tensor([b[2] for b in batch], device=device)
            with amp_ctx():
                logits = model(**enc).logits
            loss = F.cross_entropy(logits.float(), labels) / accum
            scaler.scale(loss).backward()
            run_loss = 0.98 * run_loss + 0.02 * loss.item() * accum if bi else loss.item() * accum
            pending += 1
            if pending == accum or s + micro_bs >= len(order):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
                pending = 0
            if bi % 20 == 0:
                pbar.set_postfix(loss=round(run_loss, 4))
    model.eval()
    return model, tok


@torch.no_grad()
def score_pairs(bundle, pairs, max_len, bs, desc="推論"):
    """pairs: [(text_a, text_b), ...] -> logits [n, num_labels]（依長度排序分批以加速）。"""
    model, tok = bundle
    model.eval()
    order = np.argsort([len(a) + len(b) for a, b in pairs], kind="stable")
    out = None
    for s in tqdm(range(0, len(order), bs), desc=desc, mininterval=10):
        ids = order[s:s + bs]
        enc = tok([pairs[i][0] for i in ids], [pairs[i][1] for i in ids], truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with amp_ctx():
            lg = model(**enc).logits.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(pairs), lg.shape[1]), dtype=np.float32)
        out[ids] = lg
    return out if out is not None else np.zeros((0, 2), dtype=np.float32)


def offload(bundle):
    """把不再使用的模型移到 CPU，釋放 4GB 顯存給下一階段的訓練。"""
    bundle[0].to("cpu")
    free_gpu()


def save_bundle(bundle, path):
    bundle[0].save_pretrained(path)
    bundle[1].save_pretrained(path)


def load_bundle(path):
    return (AutoModelForSequenceClassification.from_pretrained(path).to(device).eval(), AutoTokenizer.from_pretrained(path))


# ----- 小型 LLM 分類器（Qwen3-0.6B-Base + LoRA）-----
# 解碼器模型用「最後一個 token」做分類，所以把輸入寫成：證據 → 論述 → 提示語；超長時只截證據，確保 claim 與提示語完整。
def llm_parts(claim, ctx):
    pre = "證據：\n" + ctx.replace(" [SEP] ", "\n")
    post = "\n\n論述：" + claim + "\n論述與證據的關係（支持／反駁／無法判斷）："
    return pre, post


def llm_ids(tok, pre, post, max_len):
    a = tok(pre, add_special_tokens=False)["input_ids"]
    b = tok(post, add_special_tokens=False)["input_ids"]
    if len(a) + len(b) > max_len:
        a = a[:max(16, max_len - len(b))]
    return (a + b)[-max_len:]


def _llm_pad(batch_ids, pad_id):
    L = max(len(x) for x in batch_ids)
    ids = torch.full((len(batch_ids), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(batch_ids), L), dtype=torch.long)
    for i, x in enumerate(batch_ids):
        ids[i, :len(x)] = torch.tensor(x)
        att[i, :len(x)] = 1
    return ids.to(device), att.to(device)


def _llm_base(name, pad_id):
    try:
        base = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3, dtype=torch.bfloat16)
    except (TypeError, ValueError):
        base = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3, torch_dtype=torch.bfloat16)
    base.config.pad_token_id = pad_id
    return base


def _pad_id(tok):
    return tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id


def train_llm_clf(examples, model_name, max_len, epochs, lr, micro_bs, accum, desc, seed=None, lora_r=16):
    """examples: [(claim, ctx_text, label)]。基底權重 bf16 凍結 + LoRA（fp32），bf16 autocast，不需要 GradScaler。"""
    try:
        from peft import LoraConfig, get_peft_model
    except ImportError as e:
        raise ImportError("需要 peft：在 WSL 的 venv 執行  pip install peft  後重新啟動 kernel") from e
    seed = SEED if seed is None else seed
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    pad_id = _pad_id(tok)
    base = _llm_base(model_name, pad_id)
    if GRAD_CKPT:
        try:
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            base.gradient_checkpointing_enable()
        base.enable_input_require_grads()
    cfg = LoraConfig(task_type="SEQ_CLS", r=lora_r, lora_alpha=2 * lora_r, lora_dropout=0.05,
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(base, cfg).to(device)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()                           # LoRA 與分類頭用 fp32 主權重，避免 bf16 吃掉小的更新量
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"  可訓練參數 {sum(p.numel() for p in params) / 1e6:.1f}M / 總參數 {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    steps = math.ceil(len(examples) / (micro_bs * accum)) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    rg = random.Random(seed)
    for ep in range(epochs):
        model.train()
        order = list(range(len(examples)))
        rg.shuffle(order)
        pbar = tqdm(range(0, len(order), micro_bs), desc=f"{desc} epoch {ep + 1}/{epochs}", mininterval=10)
        run_loss, pending = 0.0, 0
        for bi, s_ in enumerate(pbar):
            batch = [examples[i] for i in order[s_:s_ + micro_bs]]
            ids, att = _llm_pad([llm_ids(tok, *llm_parts(b[0], b[1]), max_len) for b in batch], pad_id)
            labels = torch.tensor([b[2] for b in batch], device=device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                logits = model(input_ids=ids, attention_mask=att).logits
            loss = F.cross_entropy(logits.float(), labels) / accum
            loss.backward()
            run_loss = 0.98 * run_loss + 0.02 * loss.item() * accum if bi else loss.item() * accum
            pending += 1
            if pending == accum or s_ + micro_bs >= len(order):
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
                pending = 0
            if bi % 20 == 0:
                pbar.set_postfix(loss=round(run_loss, 4))
    model.eval()
    return model, tok


def save_llm(bundle, path):
    bundle[0].save_pretrained(path)                           # 只存 LoRA 轉接器與分類頭（很小）
    bundle[1].save_pretrained(path)


def load_llm(path, base_name):
    from peft import PeftModel
    tok = AutoTokenizer.from_pretrained(path)
    model = PeftModel.from_pretrained(_llm_base(base_name, _pad_id(tok)), path).merge_and_unload()
    return model.to(device).eval(), tok


@torch.no_grad()
def score_llm(bundle, parts, max_len, bs, desc="推論"):
    model, tok = bundle
    pad_id = _pad_id(tok)
    seqs = [llm_ids(tok, pre, post, max_len) for pre, post in parts]
    order = np.argsort([len(x) for x in seqs], kind="stable")
    out = np.zeros((len(seqs), 3), dtype=np.float32)
    for s_ in tqdm(range(0, len(order), bs), desc=desc, mininterval=10):
        idx = order[s_:s_ + bs]
        ids, att = _llm_pad([seqs[i] for i in idx], pad_id)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            out[idx] = model(input_ids=ids, attention_mask=att).logits.float().cpu().numpy()
    return out

## 2. Document Retrieval：找相關頁面

每筆標準證據是 `[annotation_id, evidence_id, page_title, sentence_id]`。
先收集公開訓練資料涉及的頁面，再從 Wikipedia 讀取這些頁面的文字。
本範例只索引這些頁面，因此找不到的頁面會限制後續表現。



In [5]:
def scan_wiki(path):
    titles, offsets, plen, leads = [], [], [], []
    pos = 0
    with open(path, "rb") as f:
        for raw in tqdm(f, desc="掃描 wiki", unit="頁", mininterval=5):
            n = len(raw)
            if raw.strip():
                page = json.loads(raw)
                t = page["id"]
                text = page.get("text", "") or ""
                titles.append(t)
                offsets.append(pos)
                plen.append(len(text))
                leads.append(title_base(t) + "：" + norm(text[:LEAD_RAW_CHARS])[:LEAD_CHARS])
            pos += n
    return dict(titles=titles, offsets=np.array(offsets, dtype=np.int64), plen=np.array(plen, dtype=np.int32), leads=leads)


_sig = f"{WIKI_PATH.stat().st_size}_{LEAD_CHARS}"
wiki = cached(CACHE_DIR / f"wiki_meta_{_sig}.pkl", lambda: scan_wiki(WIKI_PATH))
titles, offsets, plen, leads = wiki["titles"], wiki["offsets"], wiki["plen"], wiki["leads"]
NPAGES = len(titles)
title2idx = {t: i for i, t in enumerate(titles)}
base_titles = [title_base(t) for t in titles]
base2idx = defaultdict(list)           # '愛爾蘭' -> [頁面索引...]；標題本身（沒有括號）排最前面
for i, (t, b) in enumerate(zip(titles, base_titles)):
    if len(b) >= TITLE_MIN:
        if t.replace("_", " ") == b:
            base2idx[b].insert(0, i)
        else:
            base2idx[b].append(i)
print("頁面數：", NPAGES, "| 可比對的標題數：", len(base2idx))


if QUICK_TEST:   # 抽樣 wiki 只含部分頁面：只保留證據頁面都在抽樣內的題目，並重新建立 train / dev / test（完整模式不會進到這裡）
    def covered(groups):
        return any(all(t in title2idx for t, _ in g) for g in groups)

    rng = random.Random(SEED)
    ok = lambda r: r["label"].upper() == "NOT ENOUGH INFO" or covered(parse_gold(r))
    train_rows, dev_rows = [r for r in train_rows if ok(r)], [r for r in dev_rows if ok(r)]
    for rows, lim in ((train_rows, N_TRAIN_LIMIT), (dev_rows, N_DEV_LIMIT), (test_rows, N_TEST_LIMIT)):
        if lim is not None and len(rows) > lim:
            rows[:] = rng.sample(rows, lim)
    train, dev, test = make_split(train_rows, "train"), make_split(dev_rows, "dev"), make_split(test_rows, "test", labeled=False)


missing = sum(1 for s in (train, dev) for gs in s["gold"] for g in gs for t, _ in g if t not in title2idx)
print("標準答案頁面不在 wiki 的數量（完整模式應為 0；QUICK 模式因 wiki 為抽樣，可能不為 0）：", missing)

頁面數： 1187751 | 可比對的標題數： 1078608
標準答案頁面不在 wiki 的數量（完整模式應為 0；QUICK 模式因 wiki 為抽樣，可能不為 0）： 0


### 用 BERT 將文字轉成向量

BERT 為每個 token 產生一個向量，這裡用 **mean pooling** 將它們平均成文字向量。
平均時使用 attention mask 排除 padding，避免補齊長度的 token 影響結果。
再用 **L2 normalization** 把向量長度變成 1，這樣內積就等於 cosine similarity。

頁面與 claim 使用同一個預訓練 BERT，這個檢索模型不參與後面的分類器訓練。
每頁只編碼標題與開頭文字，超過 `RETRIEVAL_MAX_LENGTH` 的 token 會被截斷。
同一個 encode_texts 函式會用於頁面、claim 和證據句的編碼。
原始 BERT 未針對檢索做對比學習訓練，這裡用來示範向量檢索。



In [6]:
# ----- BM25（字元 bigram，hashing）-----
HV = HashingVectorizer(analyzer="char", ngram_range=(2, 2), n_features=BM25_FEATURES, alternate_sign=False,
                       norm=None, lowercase=True, dtype=np.float32)


def build_bm25():
    mats = [HV.transform(leads[s:s + 100000]) for s in tqdm(range(0, NPAGES, 100000), desc="BM25 建索引")]
    X = sp.vstack(mats, format="csr")
    del mats
    df = np.bincount(X.indices, minlength=BM25_FEATURES)
    idf = np.log((NPAGES - df + 0.5) / (df + 0.5) + 1).astype(np.float32)
    idf[df > BM25_MAX_DF * NPAGES] = 0.0          # 幾乎每頁都有的 bigram（停用詞）直接忽略
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    dl_rep = np.repeat(dl, np.diff(X.indptr))
    X.data = X.data * (BM25_K1 + 1) / (X.data + BM25_K1 * (1 - BM25_B + BM25_B * dl_rep / max(dl.mean(), 1e-6)))
    del dl_rep
    return X.tocsc(), idf


def bm25_scores(claim):
    q = HV.transform([claim])
    scores = np.zeros(NPAGES, dtype=np.float32)
    for c in q.indices:
        w = BM25_IDF[c]
        if w > 0:
            s, e = BM25_PTR[c], BM25_PTR[c + 1]
            scores[BM25_IND[s:e]] += w * BM25_DAT[s:e]
    return scores


def topk_idx(scores, k):
    k = min(k, len(scores))
    top = np.argpartition(-scores, k - 1)[:k]
    return top[np.argsort(-scores[top], kind="stable")]


@torch.no_grad()
def encode_dense(texts, tok, model, desc="編碼"):
    """CLS 向量 + L2 正規化，回傳 float16 numpy。依長度排序後分批，可省很多時間。"""
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float16)
    for s in tqdm(range(0, len(texts), DENSE_BS), desc=desc, mininterval=5):
        ids = order[s:s + DENSE_BS]
        enc = tok([texts[i] for i in ids], padding=True, truncation=True, max_length=DENSE_MAX_LEN, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        h = model(**enc).last_hidden_state[:, 0]
        out[ids] = F.normalize(h.float(), dim=-1).cpu().numpy().astype(np.float16)
    return out


@torch.no_grad()
def dense_topk(Q, P, k, chunk=200000, qbatch=512):
    """Q:[n,d]、P:[N,d]（皆 float16 numpy）。分塊上 GPU 算內積並合併 top-k，避免一次吃光 4GB 顯存。"""
    n, k = len(Q), min(k, len(P))
    dt = torch.float16 if device.type == "cuda" else torch.float32
    Qg = torch.from_numpy(np.ascontiguousarray(Q)).to(device=device, dtype=dt)
    best_v = torch.full((n, k), -2.0, device=device)
    best_i = torch.zeros((n, k), dtype=torch.long, device=device)
    for ps in tqdm(range(0, len(P), chunk), desc="稠密檢索", mininterval=5):
        Pc = torch.from_numpy(np.array(P[ps:ps + chunk])).to(device=device, dtype=dt)
        for qs in range(0, n, qbatch):
            v, i = (Qg[qs:qs + qbatch] @ Pc.T).topk(min(k, Pc.shape[0]), dim=1)
            cv = torch.cat([best_v[qs:qs + qbatch], v.float()], 1)
            ci = torch.cat([best_i[qs:qs + qbatch], i + ps], 1)
            tv, ti = cv.topk(k, dim=1)
            best_v[qs:qs + qbatch], best_i[qs:qs + qbatch] = tv, ci.gather(1, ti)
    return best_i.cpu().numpy(), best_v.cpu().numpy()

### 建立頁面向量索引

每個頁面只需要編碼一次，之後所有 claim 都與這些固定向量比較。
此處的索引就是一個矩陣，每一列對應 `page_titles` 中的一個頁面。
BERT-base 的向量維度為 768，索引保留在 CPU，以節省 GPU 記憶體。



In [7]:
_bm25_path = CACHE_DIR / f"bm25_{_sig}.npz"
if _bm25_path.exists():
    BM25_X, BM25_IDF = sp.load_npz(_bm25_path), np.load(CACHE_DIR / f"bm25_idf_{_sig}.npy")
else:
    BM25_X, BM25_IDF = build_bm25()
    sp.save_npz(_bm25_path, BM25_X, compressed=False)
    np.save(CACHE_DIR / f"bm25_idf_{_sig}.npy", BM25_IDF)
BM25_PTR, BM25_IND, BM25_DAT = BM25_X.indptr, BM25_X.indices, BM25_X.data
print("BM25 索引：", BM25_X.shape, "非零元素：", BM25_X.nnz)


_DENSE = {}


def dense_state():
    """第一次需要稠密檢索時才載入模型；頁面向量只算一次並存檔（約 10~20 分鐘）。"""
    if not _DENSE:
        tok, mdl = load_dense_model()
        path = CACHE_DIR / f"dense_pages_{_sig}_{DENSE_MODEL.split('/')[-1]}.npy"
        if not path.exists():
            np.save(path, encode_dense(leads, tok, mdl, desc="編碼全部頁面"))
        _DENSE.update(tok=tok, mdl=mdl, P=np.load(path, mmap_mode="r"))
        print("頁面向量：", _DENSE["P"].shape)
    return _DENSE


dense_state()

BM25 索引： (1187751, 2097152) 非零元素： 84250765



Loading weights:   0%|          | 0/71 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 71/71 [00:00<00:00, 2688.15it/s]

頁面向量： (1187751, 512)


{'tok': BertTokenizer(name_or_path='BAAI/bge-small-zh-v1.5', vocab_size=21128, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
 	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 }),
 'mdl': BertModel(
   (embeddings): BertEmbeddings(
     (word_embeddings): Embedding(21128, 512, padding_idx=0)
     (position_embeddings): Embedding(512, 512)


### 用 cosine similarity 找最相關的頁面

把 claim 也轉成單位向量，再計算 `claim_embeddings @ document_embeddings.T`。
結果矩陣的每一列是一個 claim，每一欄是一個頁面；數值越大，向量方向越接近。
`topk` 選出每個 claim 相似度最高的頁面。Cosine similarity 可為負數，仍依排名取前 K 個。
claim 向量只需計算一次，下一階段選句時會重用。



In [8]:
# ----- 標題比對 + 候選頁面 + 特徵 -----
FEATS = ["tm", "tlen", "tfrac", "tpos", "n_tm", "tm_rank", "nested", "paren", "same_base", "digit", "ptlen",
         "bm25", "bm25_rel", "bm25_rank", "dense", "dense_rel", "dense_rank", "plen", "lead_ov"]
FI = {f: i for i, f in enumerate(FEATS)}


def title_matches(claim):
    """claim 中所有以子字串形式出現的 wiki 標題（已去括號、長度 >= 2）。"""
    L, found = len(claim), {}
    for i in range(L):
        for j in range(i + TITLE_MIN, min(L, i + TITLE_MAX) + 1):
            sub = claim[i:j]
            if sub in base2idx and sub not in found:
                found[sub] = (i, j)
    return [(b, s, e) for b, (s, e) in found.items()]


def gen_features(claims, Q, d_idx, d_val, desc):
    rows_all, pidx_all, counts = [], [], []
    for qi, claim in enumerate(tqdm(claims, desc=desc, mininterval=5)):
        sc = bm25_scores(claim)
        top = topk_idx(sc, K_BM25)
        b_rank = {int(i): r for r, i in enumerate(top)}
        bmax = float(sc[top[0]]) if len(top) else 0.0
        if USE_DENSE:
            DENSE_P = dense_state()["P"]
            dq, di, dv = Q[qi].astype(np.float32), d_idx[qi], d_val[qi]
            d_rank = {int(i): r for r, i in enumerate(di)}
            dmax = float(dv[0])
        else:
            di, d_rank, dmax = np.array([], dtype=np.int64), {}, 0.0
        tms = sorted(title_matches(claim), key=lambda x: -(x[2] - x[1]))
        n_tm = len(tms)
        tm_rank_of = {b: r for r, (b, _, _) in enumerate(tms)}
        bases = [b for b, _, _ in tms]
        nested_of = {b: int(any(o != b and b in o for o in bases)) for b in bases}
        info = {}
        for b, s, e in tms:
            for pidx in base2idx[b][:PAGES_PER_TITLE]:
                if pidx not in info and len(info) < MAX_TITLE_CANDS:
                    info[pidx] = (b, s, e)
        cset = dict.fromkeys(info)
        for i in top[:CAND_BM25]:
            cset[int(i)] = None
        for i in di[:CAND_DENSE]:
            cset[int(i)] = None
        cl = list(cset)
        cos = {}
        if USE_DENSE:
            cos = {i: float(dv[d_rank[i]]) for i in cl if i in d_rank}
            miss = [i for i in cl if i not in d_rank]
            if miss:
                vals = np.asarray(DENSE_P[np.array(miss)], dtype=np.float32) @ dq
                cos.update({i: float(v) for i, v in zip(miss, vals)})
        cb = bigrams(claim)
        den, L = max(1, len(cb)), max(1, len(claim))
        rows = []
        for i in cl:
            inf = info.get(i)
            if inf:
                b, s, e = inf
                tm, tl, tf, tp, tr, ne, dg = 1, e - s, (e - s) / L, s / L, tm_rank_of[b], nested_of[b], int(b.isdigit())
            else:
                tm, tl, tf, tp, tr, ne, dg = 0, 0, 0.0, -1.0, n_tm, 0, 0
            bm = float(sc[i])
            dn = cos.get(i, 0.0)
            rows.append([tm, tl, tf, tp, n_tm, tr, ne, int("_(" in titles[i]), len(base2idx.get(base_titles[i], ())), dg,
                         len(base_titles[i]), bm, bm / bmax if bmax > 0 else 0.0, b_rank.get(i, K_BM25),
                         dn, dn - dmax, d_rank.get(i, K_DENSE), math.log1p(plen[i]), len(cb & bigrams(leads[i])) / den])
        rows_all.append(np.asarray(rows, dtype=np.float32).reshape(-1, len(FEATS)))
        pidx_all.append(np.asarray(cl, dtype=np.int64))
        counts.append(len(cl))
    counts = np.array(counts)
    return dict(X=np.concatenate(rows_all), pidx=np.concatenate(pidx_all), counts=counts,
                starts=np.concatenate([[0], np.cumsum(counts)[:-1]]))


def retrieve_features(split):
    path = CACHE_DIR / f"feat_{_sig}_{split['name']}_{len(split['claim'])}.pkl"

    def build():
        if USE_DENSE:
            ds = dense_state()
            Q = encode_dense(split["claim"], ds["tok"], ds["mdl"], desc=f"編碼 {split['name']} claim")
            d_idx, d_val = dense_topk(Q, ds["P"], K_DENSE)
        else:
            Q = d_idx = d_val = None
        return gen_features(split["claim"], Q, d_idx, d_val, desc=f"候選頁面 {split['name']}")
    return cached(path, build)


feats = {s["name"]: retrieve_features(s) for s in (dev, test, train)}
_DENSE.pop("mdl", None)         # 稠密模型之後不再需要，釋放顯存給 BERT 訓練
free_gpu()
for k, v in feats.items():
    print(k, "候選列數：", len(v["X"]), "每題平均候選頁數：", round(len(v["X"]) / len(v["counts"]), 1))


# ----- 學習式排序（HistGradientBoosting）與 Recall@K -----
def page_labels(split, F_):
    y = np.zeros(len(F_["X"]), dtype=np.int8)
    for qi, groups in enumerate(split["gold"]):
        gold = {title2idx[t] for g in groups for t, _ in g if t in title2idx}
        s, c = F_["starts"][qi], F_["counts"][qi]
        y[s:s + c] = [int(p in gold) for p in F_["pidx"][s:s + c]]
    return y


def fit_ranker(X, y):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_leaf_nodes=48, l2_regularization=1.0,
                                          random_state=SEED).fit(X, y)


def to_ranked(F_, scores):
    out = []
    for qi in range(len(F_["counts"])):
        s, c = F_["starts"][qi], F_["counts"][qi]
        o = np.argsort(-scores[s:s + c], kind="stable")
        out.append((F_["pidx"][s:s + c][o], scores[s:s + c][o]))
    return out


Ftr = feats["train"]
y_tr = page_labels(train, Ftr)
qi_rows = np.repeat(np.arange(len(Ftr["counts"])), Ftr["counts"])
sr_rows = np.where(train["label"][qi_rows] != 2)[0]
nei_rows = np.where(train["label"][qi_rows] == 2)[0]
ranker = fit_ranker(Ftr["X"][sr_rows], y_tr[sr_rows])
oof = np.zeros(len(Ftr["X"]), dtype=np.float32)
if len(nei_rows):
    oof[nei_rows] = ranker.predict_proba(Ftr["X"][nei_rows])[:, 1]
for tr_i, va_i in GroupKFold(n_splits=3).split(sr_rows, groups=qi_rows[sr_rows]):   # 訓練題用交叉預測，避免排序分數過度樂觀
    m = fit_ranker(Ftr["X"][sr_rows[tr_i]], y_tr[sr_rows[tr_i]])
    oof[sr_rows[va_i]] = m.predict_proba(Ftr["X"][sr_rows[va_i]])[:, 1]
ranked = {"train": to_ranked(Ftr, oof)}
for name in ("dev", "test"):
    ranked[name] = to_ranked(feats[name], ranker.predict_proba(feats[name]["X"])[:, 1])


print("測試 claim：", test["claim"][0])
print("取回頁面：", [titles[i] for i in ranked["test"][0][0][:PAGE_TOPK]])

dev 候選列數： 160680 每題平均候選頁數： 53.6
test 候選列數： 334376 每題平均候選頁數： 55.7
train 候選列數： 1279826 每題平均候選頁數： 53.3


測試 claim： 沈懋孝爲明朝的平民，對書籍厭惡，毀壞數量達萬卷。
取回頁面： ['沈懋孝', '厭惡', '明朝', '沈懋學', '祕魯鰹鳥', '莊肅']


### Document Retrieval Recall@5

用有標準答案的驗證集計分，只統計 supports、refutes 題目。
前 5 個頁面必須包含任一完整證據組需要的所有頁面，該題才算命中。
若一組證據需要 A、B 兩頁，只找到 A 仍算未命中。此分數不考慮分類結果。



In [9]:
def page_recall(split, rk, K):
    hit = n = 0
    for qi, groups in enumerate(split["gold"]):
        if not groups:
            continue
        n += 1
        top = {titles[i] for i in rk[qi][0][:K]}
        hit += any(all(t in top for t, _ in g) for g in groups)
    return hit / max(n, 1)


# 消融：各訊號單獨排序 vs. 學習式排序（候選集相同，所以 K<=20 時 BM25 / 稠密單獨排序等同於原本的 top-K）
Fd, X = feats["dev"], feats["dev"]["X"]
rrf = 1 / (60 + X[:, FI["bm25_rank"]]) + 1 / (60 + X[:, FI["dense_rank"]]) + X[:, FI["tm"]] / (60 + X[:, FI["tm_rank"]])
variants = {
    "只用標題比對": X[:, FI["tm"]] * (1 + X[:, FI["tlen"]]) + 1e-3 * X[:, FI["bm25_rel"]],
    "只用 BM25": X[:, FI["bm25"]],
    "只用稠密檢索": X[:, FI["dense"]] if USE_DENSE else np.zeros(len(X)),
    "RRF 融合（無學習）": rrf,
    "學習式排序（本方法）": None,
}
Ks = (1, 3, 5, 6, 10, 20)
page_table = {}
for name, sc in variants.items():
    rk = ranked["dev"] if sc is None else to_ranked(Fd, sc)
    page_table[name] = [page_recall(dev, rk, K) for K in Ks]
page_df = pd.DataFrame(page_table, index=[f"Recall@{K}" for K in Ks]).T
print("Dev 頁面檢索（完整證據組的所有頁面都在前 K 名才算命中，僅統計 supports/refutes）")
display(page_df.round(4)) if "display" in globals() else print(page_df.round(4))
cand_recall = page_recall(dev, to_ranked(Fd, np.ones(len(X))), 10 ** 9)
print("候選集本身的上限（Recall@全部候選）：", round(cand_recall, 4))

Dev 頁面檢索（完整證據組的所有頁面都在前 K 名才算命中，僅統計 supports/refutes）
             Recall@1  Recall@3  Recall@5  Recall@6  Recall@10  Recall@20
只用標題比對         0.5235    0.7405    0.8255    0.8385     0.8635     0.9000
只用 BM25        0.5400    0.6385    0.6725    0.6845     0.7175     0.7585
只用稠密檢索         0.5235    0.6575    0.6965    0.7135     0.7555     0.7975
RRF 融合（無學習）    0.6995    0.8290    0.8535    0.8570     0.8760     0.9220
學習式排序（本方法）     0.7915    0.8955    0.9175    0.9245     0.9375     0.9410
候選集本身的上限（Recall@全部候選）： 0.9425


## 3. Evidence Retrieval：從頁面挑選句子

第一階段回答「看哪些頁面」，第二階段回答「使用哪些句子」。
候選句子只使用公開訓練集標註過的句子，並保留 Wikipedia 的原始句子編號。
例如跳過空白的第 1 句後，第 2 句仍然是第 2 句，不可以重新編號。
同樣用 BERT、mean pooling 和 L2 normalization 建立句子向量，再算 cosine similarity。



In [10]:
WIKI_FH = open(WIKI_PATH, "rb")


@functools.lru_cache(maxsize=80000)
def get_sentences(pidx):
    """回傳 ((sentence_id, 去空白文字, 超連結目標頁面tuple), ...)；略過空句。"""
    WIKI_FH.seek(int(offsets[pidx]))
    page = json.loads(WIKI_FH.readline())
    out = []
    for line in page.get("lines", "").split("\n"):
        parts = line.split("\t")
        if len(parts) < 2 or not parts[0].isdigit():
            continue
        text = norm(parts[1])
        if text:
            out.append((int(parts[0]), text, tuple(x for x in parts[3::2] if x)))   # parts[2::2] 是錨文字，[3::2] 是目標頁面
    return tuple(out)


def disp(pidx):
    return titles[pidx].replace("_", " ")


def passage(pidx, text):
    return f"{disp(pidx)}：{text}"


print(get_sentences(0)[:2])

((0, '數學，是研究數量、結構以及空間等概念及其變化的一門學科，屬於形式科學的一種。', ()), (2, '基礎數學的知識與運用是生活中不可或缺的一環。', ()))


In [11]:
def build_pool(claim, page_idx, page_prob, pool_size):
    """回傳依粗篩分數排序的候選句 [(pre, pidx, sid, text, links), ...]。"""
    cb = bigrams(claim)
    den = max(1, len(cb))
    cands = []
    for pidx, pp in zip(page_idx, page_prob):
        for sid, text, links in get_sentences(int(pidx)):
            if sid >= PER_PAGE_MAX_SID:
                continue
            cands.append((len(cb & bigrams(text)) / den + 0.5 * float(pp) + (0.1 if sid == 0 else 0.0), int(pidx), sid, text, links))
    cands.sort(key=lambda c: -c[0])
    return cands[:pool_size]


def extra_pool(claim, pages, per_page, exclude=()):
    """指定頁面中，粗篩分數最高的句子（用於第二跳與訓練負例）。"""
    cb = bigrams(claim)
    den = max(1, len(cb))
    cands = []
    for pidx in pages:
        sents = [(len(cb & bigrams(t)) / den + (0.1 if sid == 0 else 0.0), int(pidx), sid, t, lk)
                 for sid, t, lk in get_sentences(int(pidx)) if sid < PER_PAGE_MAX_SID and (titles[int(pidx)], sid) not in exclude]
        sents.sort(key=lambda c: -c[0])
        cands.extend(sents[:per_page])
    cands.sort(key=lambda c: -c[0])
    return cands


# ----- 訓練 cross-encoder 重排器 -----
def build_rerank_examples(split, rk, claim_ids):
    rg = random.Random(SEED)
    ex = []
    for qi in tqdm(claim_ids, desc="建立重排訓練資料", mininterval=5):
        groups = split["gold"][qi]
        claim = split["claim"][qi]
        gold = {(t, s) for g in groups for t, s in g}
        pos, gold_pages = [], []
        for t, s in gold:
            j = title2idx.get(t)
            if j is None:
                continue
            gold_pages.append(j)
            for sid, text, links in get_sentences(j):
                if sid == s:
                    pos.append((claim, passage(j, text), 1))
        if not pos:
            continue
        idxs, probs = rk[qi]
        pool = build_pool(claim, idxs[:PAGE_TOPK], probs[:PAGE_TOPK], 40)
        negs = {(c[1], c[2]): c for c in pool if (titles[c[1]], c[2]) not in gold}
        for c in extra_pool(claim, set(gold_pages), 6, exclude=gold):                     # 同一個標準頁面的其他句子
            negs.setdefault((c[1], c[2]), c)
        hop_pages = set()
        for t, s in gold:
            j = title2idx.get(t)
            for sid, _, links in (get_sentences(j) if j is not None else ()):
                if sid == s:
                    hop_pages.update(title2idx[l] for l in links if l in title2idx)
        for c in extra_pool(claim, hop_pages - set(gold_pages), 3, exclude=gold)[:6]:      # 標準證據句連結到的頁面（第二跳負例）
            negs.setdefault((c[1], c[2]), c)
        neg_list = sorted(negs.values(), key=lambda c: -c[0])
        chosen = neg_list[:3] + rg.sample(neg_list[3:], min(RERANK_NEG - 3, len(neg_list[3:])))
        ex.extend(pos)
        ex.extend((claim, passage(c[1], c[3]), 0) for c in chosen)
    rg.shuffle(ex)
    return ex


_rr_path = CACHE_DIR / "reranker"
if _rr_path.exists() and not FORCE_RETRAIN:
    reranker = load_bundle(_rr_path)
    print("載入已訓練的重排器：", _rr_path)
else:
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    random.Random(SEED).shuffle(sr_ids)
    sr_ids = sr_ids[:RERANK_TRAIN_CLAIMS] if RERANK_TRAIN_CLAIMS else sr_ids
    rr_examples = build_rerank_examples(train, ranked["train"], sr_ids)
    print("重排訓練資料：", len(rr_examples), "筆（正例", sum(e[2] for e in rr_examples), "）")
    reranker = train_seq_clf(rr_examples, 2, RERANK_MODEL, RERANK_MAX_LEN, RERANK_EPOCHS, RERANK_LR, RERANK_BS, RERANK_ACCUM, "訓練重排器")
    save_bundle(reranker, _rr_path)
    del rr_examples
free_gpu()
RR_TAG = str(int((_rr_path / "config.json").stat().st_mtime))   # 重排器重訓後，舊的重排快取自動失效


# ----- 重排推論（含第二跳）-----
def rerank_split(split, rk, desc):
    """回傳 (first, final)：first 只含第一跳；final 加上第二跳。每題是 [(score, pidx, sid, text, links), ...] 由高到低。"""
    n = len(split["claim"])
    pools = [build_pool(split["claim"][qi], rk[qi][0][:PAGE_TOPK], rk[qi][1][:PAGE_TOPK], RERANK_POOL) for qi in range(n)]
    pairs = [(split["claim"][qi], passage(c[1], c[3])) for qi in range(n) for c in pools[qi]]
    own = [(qi, k) for qi in range(n) for k in range(len(pools[qi]))]
    lg = score_pairs(reranker, pairs, RERANK_MAX_LEN, INFER_BS, desc)
    first = [[] for _ in range(n)]
    sc = (lg[:, 1] - lg[:, 0]) if len(lg) else np.zeros(0)
    for (qi, k), v in zip(own, sc):
        c = pools[qi][k]
        first[qi].append((float(v), c[1], c[2], c[3], c[4]))
    first = [sorted(x, key=lambda t: -t[0]) for x in first]
    if not USE_HOP2:
        return first, first
    hop_pools, hop_pairs, hop_own = [], [], []
    for qi in range(n):
        have = {c[1] for c in pools[qi]}
        targets = []
        for _, pidx, sid, text, links in first[qi][:HOP_SRC]:
            for t in links:
                j = title2idx.get(t)
                if j is not None and j not in have and j not in targets:
                    targets.append(j)
        hp = extra_pool(split["claim"][qi], targets[:HOP_MAX_PAGES], HOP_PER_PAGE)[:HOP_POOL]
        hop_pools.append(hp)
        for k, c in enumerate(hp):
            hop_pairs.append((split["claim"][qi], passage(c[1], c[3])))
            hop_own.append((qi, k))
    final = [list(x) for x in first]
    if hop_pairs:
        hl = score_pairs(reranker, hop_pairs, RERANK_MAX_LEN, INFER_BS, desc + "（第二跳）")
        for (qi, k), v in zip(hop_own, hl[:, 1] - hl[:, 0]):
            c = hop_pools[qi][k]
            final[qi].append((float(v), c[1], c[2], c[3], c[4]))
    final = [sorted(x, key=lambda t: -t[0]) for x in final]
    return first, final


def evidence_recall(split, lists, K=5):
    hit = n = 0
    for qi, groups in enumerate(split["gold"]):
        if not groups:
            continue
        n += 1
        top = {(titles[c[1]], c[2]) for c in lists[qi][:K]}
        hit += any(all(x in top for x in g) for g in groups)
    return hit / max(n, 1)


rr = {}
for s in (dev, test):
    rr[s["name"]] = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_{s['name']}_{len(s['claim'])}.pkl",
                           lambda s=s: rerank_split(s, ranked[s["name"]], f"重排 {s['name']}"))


def merge_hop(first, final, delta):
    """第二跳候選（final 中不在 first 的句子）扣掉 delta 分後與第一跳合併；delta=None 代表不使用第二跳。"""
    if delta is None:
        return first
    seen = {(c[1], c[2]) for c in first}
    out = list(first) + [(c[0] - delta,) + tuple(c[1:]) for c in final if (c[1], c[2]) not in seen]
    return sorted(out, key=lambda t: -t[0])


ev_rows = {}
for d in HOP_PENALTIES:
    lists = [merge_hop(a, b, d) for a, b in zip(*rr["dev"])]
    ev_rows["不使用第二跳" if d is None else f"第二跳（扣 {d:g} 分）"] = [evidence_recall(dev, lists, K) for K in (1, 3, 5)]
ev_df = pd.DataFrame(ev_rows, index=["Recall@1", "Recall@3", "Recall@5"]).T
print("Dev 證據句檢索（前 K 句需完整涵蓋任一組標準證據；僅統計 supports/refutes）")
display(ev_df.round(4)) if "display" in globals() else print(ev_df.round(4))

# 第二跳的句子容易擠掉第一跳的正確句子，所以只有在 Recall@5 明顯提升（>= 0.2 個百分點）時才採用
HOP_DELTA, _best = None, evidence_recall(dev, rr["dev"][0], 5)
for d in HOP_PENALTIES:
    if d is not None and ev_rows[f"第二跳（扣 {d:g} 分）"][2] > _best + 0.002:
        HOP_DELTA, _best = d, ev_rows[f"第二跳（扣 {d:g} 分）"][2]
print("採用的第二跳扣分：", HOP_DELTA, "（None = 不使用第二跳）")
EV = {n: [merge_hop(a, b, HOP_DELTA) for a, b in zip(*rr[n])] for n in ("dev", "test")}   # 之後都使用這份證據排序
for ng in (1, 2, 3):   # 依標準證據組的句子數分析
    ids = [i for i, gs in enumerate(dev["gold"]) if gs and min(len(g) for g in gs) == ng]
    if ids:
        sub = dict(gold=[dev["gold"][i] for i in ids])
        print(f"  最短證據組含 {ng} 句：{len(ids)} 題，Recall@5 =", round(evidence_recall(sub, [EV["dev"][i] for i in ids], 5), 4))


USE_RR2 = True
RR2_MODEL = "hfl/chinese-roberta-wwm-ext"
RR2_MAX_LEN, RR2_EPOCHS, RR2_LR, RR2_BS, RR2_ACCUM = 160, 2, 3e-5, 16, 2
RR2_HARD, RR2_RAND, RR2_SIB = 3, 2, 2                 # 每題負例：第一個重排器的前 3 名非標準句 + 2 個隨機 + 同頁 2 句
SEL_N = 12                                            # 選擇器只重新排序重排器的前 12 名；第二個重排器也只替這 12 句打分


def build_rerank2_examples(ids, lists):
    rg = random.Random(SEED + 7)
    ex = []
    for pos_, qi in enumerate(tqdm(ids, desc="建立重排器2訓練資料", mininterval=5)):
        claim = train["claim"][qi]
        gold = {(t, s) for g in train["gold"][qi] for t, s in g}
        pos_txt, gold_pages = [], []
        for t, s in gold:
            j = title2idx.get(t)
            if j is None:
                continue
            gold_pages.append(j)
            pos_txt += [passage(j, text) for sid, text, _ in get_sentences(j) if sid == s]
        if not pos_txt:
            continue
        negs = [c for c in lists[pos_] if (titles[c[1]], c[2]) not in gold]
        chosen = {(c[1], c[2]): c for c in negs[:RR2_HARD]}                                   # hard negatives：第一個重排器最愛的錯誤答案
        rest = negs[RR2_HARD:RR2_HARD + 12]
        for c in rg.sample(rest, min(RR2_RAND, len(rest))):
            chosen.setdefault((c[1], c[2]), c)
        for c in extra_pool(claim, set(gold_pages), RR2_SIB, exclude=gold)[:RR2_SIB]:         # 標準證據同頁的其他句子
            chosen.setdefault((c[1], c[2]), c)
        ex.extend((claim, p, 1) for p in pos_txt)
        ex.extend((claim, passage(c[1], c[3]), 0) for c in chosen.values())
    rg.shuffle(ex)
    return ex


if USE_RR2:
    _rr2_path = CACHE_DIR / "reranker2"
    if _rr2_path.exists() and not FORCE_RETRAIN:
        reranker2 = load_bundle(_rr2_path)
        print("載入已訓練的重排器2：", _rr2_path)
    else:
        _ids2 = [i for i in range(len(train["claim"])) if train["label"][i] != 2]            # 與分類訓練資料相同的題目集合 → 共用第一個重排器的重排快取
        if CLS_TRAIN_SR_CLAIMS:
            random.Random(SEED).shuffle(_ids2)
            _ids2 = sorted(_ids2[:CLS_TRAIN_SR_CLAIMS])
        _first, _final = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_trainSR_{len(_ids2)}.pkl",
                                lambda: rerank_split(dict(claim=[train["claim"][i] for i in _ids2]), [ranked["train"][i] for i in _ids2], "重排 train-SR"))
        _lists2 = [merge_hop(a, b, HOP_DELTA) for a, b in zip(_first, _final)]
        rr2_examples = build_rerank2_examples(_ids2, _lists2)
        print("重排器2 訓練資料：", len(rr2_examples), "筆（正例", sum(e[2] for e in rr2_examples), "）")
        offload(reranker)                                                                     # 騰出顯存給第二個重排器
        reranker2 = train_seq_clf(rr2_examples, 2, RR2_MODEL, RR2_MAX_LEN, RR2_EPOCHS, RR2_LR, RR2_BS, RR2_ACCUM, "訓練重排器2", seed=SEED + 7)
        save_bundle(reranker2, _rr2_path)
        del rr2_examples, _first, _final, _lists2
        reranker[0].to(device)                                                                # 第一個重排器還要用（若分類訓練資料沒有快取）
        free_gpu()
    RR2_TAG = str(int((_rr2_path / "config.json").stat().st_mtime))


def rr2_scores(split, lists):
    """第二個重排器替每題 lists[qi][:SEL_N] 打分（logit 差），回傳每題一個 numpy 陣列；有快取。"""
    def build():
        pairs, own = [], []
        for qi, lst in enumerate(lists):
            for k, c in enumerate(lst[:SEL_N]):
                pairs.append((split["claim"][qi], passage(c[1], c[3])))
                own.append((qi, k))
        lg = score_pairs(reranker2, pairs, RR2_MAX_LEN, INFER_BS, f"重排器2 {split['name']}")
        out = [np.zeros(min(SEL_N, len(l)), dtype=np.float32) for l in lists]
        for (qi, k), v in zip(own, lg[:, 1] - lg[:, 0]):
            out[qi][k] = v
        return out
    return cached(CACHE_DIR / f"rr2score_{_sig}_{RR_TAG}_{RR2_TAG}_{HOP_DELTA}_{SEL_N}_{split['name']}_{len(lists)}.pkl", build)


if USE_RR2:
    RR2_S = {s["name"]: rr2_scores(s, EV[s["name"]]) for s in (dev, test)}
    offload(reranker2)
    free_gpu()
    _g = [(dev["gold"][qi], EV["dev"][qi][:SEL_N], RR2_S["dev"][qi]) for qi in range(len(dev["claim"]))]
    _by2 = [[c for _, c in sorted(zip(-s, lst), key=lambda t: t[0])] for _, lst, s in _g]      # 只用第二個重排器排序（診斷）
    print("Dev 證據 Recall@1/3/5（第一個重排器）：", [round(evidence_recall(dev, EV["dev"], K), 4) for K in (1, 3, 5)])
    print("Dev 證據 Recall@1/3/5（只用第二個重排器重排前 12 句）：", [round(evidence_recall(dev, _by2, K), 4) for K in (1, 3, 5)])


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2362.84it/s]

載入已訓練的重排器： output/cache/reranker


Dev 證據句檢索（前 K 句需完整涵蓋任一組標準證據；僅統計 supports/refutes）
            Recall@1  Recall@3  Recall@5
第二跳（扣 0 分）    0.4885    0.7865    0.8540
第二跳（扣 1 分）    0.4945    0.7965    0.8630
第二跳（扣 2 分）    0.4955    0.8015    0.8635
第二跳（扣 3 分）    0.4955    0.8020    0.8635
第二跳（扣 5 分）    0.4955    0.8015    0.8635
不使用第二跳        0.4955    0.8005    0.8615
採用的第二跳扣分： None （None = 不使用第二跳）
  最短證據組含 1 句：1273 題，Recall@5 = 0.9356
  最短證據組含 2 句：513 題，Recall@5 = 0.7856
  最短證據組含 3 句：169 題，Recall@5 = 0.6509



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2549.30it/s]

載入已訓練的重排器2： output/cache/reranker2


Dev 證據 Recall@1/3/5（第一個重排器）： [0.4955, 0.8005, 0.8615]
Dev 證據 Recall@1/3/5（只用第二個重排器重排前 12 句）： [0.512, 0.7875, 0.8585]


### Evidence Retrieval Recall@5

同樣只統計 supports、refutes，檢查前 5 個證據句是否完整涵蓋任一標準證據組。
每句必須同時比對頁面名稱與原始句子編號；部分命中不給分。
此分數評估實際流程（先找頁面，再選句），不假設第一階段已找到正確頁面。



In [12]:
SEL_MIN_GAIN = 0.005
N_BASE, N_LINK, N_RR2 = 21, 4, 6


def _bg(s):
    s = re.sub(r"\s+", "", s)
    return {s[i:i + 2] for i in range(len(s) - 1)}


def sel_features(split, lists, rk, s2=None):
    """回傳 (X, meta)；X 的欄位依序為 base(21) / link(4) / rr2(6)；meta 為 (題目編號, 在 lists[qi] 中的位置)。"""
    X, meta = [], []
    for qi, lst in enumerate(lists):
        cand = lst[:SEL_N]
        if not cand:
            continue
        claim = split["claim"][qi]
        cb = _bg(claim)
        rank_of = {int(p): r for r, p in enumerate(rk[qi][0][:200])}
        prob_of = {int(p): float(v) for p, v in zip(rk[qi][0][:200], rk[qi][1][:200])}
        pages = []
        for c in cand:
            if c[1] not in pages:
                pages.append(c[1])
        s0 = cand[0][0]
        s1 = cand[1][0] if len(cand) > 1 else s0 - 1.0
        top_page = cand[0][1]
        links1 = set(cand[0][4])
        links2 = set(cand[1][4]) if len(cand) > 1 else set()
        keys = {(c[1], c[2]) for c in cand}
        if s2 is not None:
            v2 = np.asarray(s2[qi], dtype=np.float64)
            r2 = np.argsort(np.argsort(-v2, kind="stable"), kind="stable")
            m2 = float(v2.max())
            m2b = float(np.sort(v2)[-2]) if len(v2) > 1 else m2 - 1.0
            p2top = cand[int(v2.argmax())][1]
        covered = set()
        for k, c in enumerate(cand):
            sc, p, sid, text = c[0], c[1], c[2], c[3]
            same = [d for d in cand if d[1] == p]
            same_sid = [d[2] for d in same]
            ttl = _PAREN.sub("", titles[p]).replace("_", " ")
            bt = _bg(text)
            inter = cb & bt
            row = [sc, k, sc - s0, sc - s1, sid, float(sid == 0), math.log1p(sid),
                   rank_of.get(int(p), 200), prob_of.get(int(p), 0.0), pages.index(p), max(d[0] for d in same) - s0,
                   sum(1 for d in cand[:5] if d[1] == p), len(same), sid - min(same_sid),
                   float(p == top_page), len(text), len(claim), float(ttl in claim), len(ttl),
                   len(inter) / max(len(cb), 1), len(inter - covered) / max(len(cb), 1)]
            row += [float(titles[p] in links1 and p != top_page), float(titles[p] in links2 and p != cand[1][1]),
                    abs(sid - cand[0][2]) if p == top_page else 99.0,
                    float((p, sid - 1) in keys) + float((p, sid + 1) in keys)]
            if s2 is not None:
                row += [v2[k], int(r2[k]), v2[k] - m2, v2[k] - m2b, float(p == p2top), (k + int(r2[k])) / 2.0]
            else:
                row += [0.0] * N_RR2
            X.append(row)
            covered |= inter
            meta.append((qi, k))
    return np.array(X, dtype=np.float32), meta


def sel_reorder(lists, meta, prob):
    """依選擇器機率重新排序前 SEL_N 名，其餘維持原順序接在後面。"""
    pq = defaultdict(dict)
    for (qi, k), p in zip(meta, prob):
        pq[qi][k] = p
    out = []
    for qi, lst in enumerate(lists):
        if qi not in pq:
            out.append(lst)
            continue
        order = sorted(pq[qi], key=lambda k: (-pq[qi][k], k))
        out.append([lst[k] for k in order] + list(lst[SEL_N:]))
    return out


def sel_model():
    return HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=30,
                                          l2_regularization=1.0, random_state=SEED)


SEL_SETS = {"base": np.arange(N_BASE), "base+link": np.arange(N_BASE + N_LINK)}
if USE_RR2:
    SEL_SETS["base+link+rr2"] = np.arange(N_BASE + N_LINK + N_RR2)
X_dev, meta_dev = sel_features(dev, EV["dev"], ranked["dev"], RR2_S["dev"] if USE_RR2 else None)
_gold_dev = [{x for g in dev["gold"][qi] for x in g} for qi in range(len(dev["claim"]))]
y_dev = np.array([(titles[EV["dev"][qi][k][1]], EV["dev"][qi][k][2]) in _gold_dev[qi] for qi, k in meta_dev])
q_dev = np.array([qi for qi, _ in meta_dev])
sr_mask = np.array([dev["label"][qi] != 2 for qi in q_dev])
_folds = list(GroupKFold(5).split(X_dev, y_dev, q_dev))
SEL_OOF, SEL_LISTS = {}, {}
for _nm, _cols in SEL_SETS.items():
    _oof = np.zeros(len(y_dev))
    for tr_i, va_i in _folds:
        tr_i = tr_i[sr_mask[tr_i]]                        # 只用 supports/refutes 題訓練（NEI 沒有標準證據）
        _oof[va_i] = sel_model().fit(X_dev[tr_i][:, _cols], y_dev[tr_i]).predict_proba(X_dev[va_i][:, _cols])[:, 1]
    SEL_LISTS[_nm] = sel_reorder(EV["dev"], meta_dev, _oof)
    SEL_OOF[_nm] = {K: round(evidence_recall(dev, SEL_LISTS[_nm], K), 4) for K in (1, 3, 5)}
print("證據句 Recall@1/3/5（dev，選擇器為 5 折 OOF）")
print("  重排器 baseline        ", {K: round(evidence_recall(dev, EV["dev"], K), 4) for K in (1, 3, 5)})
for _nm in SEL_SETS:
    print(f"  選擇器 {_nm:14s}", SEL_OOF[_nm])
_base5 = evidence_recall(dev, EV["dev"], 5)
SEL_SET, _best = None, _base5 + SEL_MIN_GAIN - 1e-9
for _nm in SEL_SETS:                                      # 由簡單到複雜；複雜的組合要多贏 0.1 個百分點才取代
    if SEL_OOF[_nm][5] >= _best + (0.001 if SEL_SET is not None else 0.0):
        SEL_SET, _best = _nm, SEL_OOF[_nm][5]
SEL_ON = SEL_SET is not None
print("採用證據句重新挑選：", SEL_ON, "| 特徵組合：", SEL_SET)
for ng in (1, 2, 3):
    ids = [i for i, gs in enumerate(dev["gold"]) if gs and min(len(g) for g in gs) == ng]
    if ids:
        sub = dict(gold=[dev["gold"][i] for i in ids])
        _sel_l = SEL_LISTS[SEL_SET] if SEL_ON else EV["dev"]
        print(f"  最短證據組 {ng} 句（{len(ids)} 題）Recall@5：",
              round(evidence_recall(sub, [EV["dev"][i] for i in ids], 5), 4), "→", round(evidence_recall(sub, [_sel_l[i] for i in ids], 5), 4))
if SEL_ON:
    _cols = SEL_SETS[SEL_SET]
    _sel_final = sel_model().fit(X_dev[sr_mask][:, _cols], y_dev[sr_mask])
    X_te, meta_te = sel_features(test, EV["test"], ranked["test"], RR2_S["test"] if USE_RR2 else None)
    EV_SEL_TEST = sel_reorder(EV["test"], meta_te, _sel_final.predict_proba(X_te[:, _cols])[:, 1])
EV_OUT = {"dev": SEL_LISTS[SEL_SET] if SEL_ON else EV["dev"], "test": EV_SEL_TEST if SEL_ON else EV["test"]}   # 只用於「送出的證據」與 CFEVER 計算


# ===== v12：證據檢索改進（2026-11-24；只用 dev 決定，見 Report Q2、Q4(c)）=====
# (1) 候選池：前 6 頁的「所有」句子（不再限制每頁前 60 句），依同一個粗篩分數取前 96 句。
#     dev 上「某一組標準證據完整落在候選池」的比例：0.9045（原本 32 句）→ 0.9245（等於頁面檢索的上限）。
# (2) 第二個打分模型：BAAI/bge-reranker-v2-m3（約 568M 參數 < 1B；別人已在外部檢索資料上訓練好的 reranker checkpoint，這裡不微調、只推論）。
# (3) 融合分數 = (1 - w)·z(原本的 MacBERT 重排器) + w·z(bge)，z 用 dev 分數的平均與標準差；再接上方同一個證據句選擇器，
#     並把其中一個原始分數放進原本 rr2 的 6 個特徵欄位。w 與放哪個分數依 dev 5 折 OOF Recall@5 選擇（與上方選擇器相同的做法）。
POOL96_CAP, RR12_MODEL, RR12_MAX_LEN = 96, "BAAI/bge-reranker-v2-m3", 192
V12_W_GRID = (0.0, 0.5, 0.75, 1.0)


def build_pool96(claim, page_idx, page_prob):
    """與 build_pool 相同的粗篩分數，但不限制句子編號，取前 POOL96_CAP 句。"""
    cb = bigrams(claim)
    den = max(1, len(cb))
    cands = []
    for pidx, pp in zip(page_idx, page_prob):
        for sid, text, links in get_sentences(int(pidx)):
            cands.append((len(cb & bigrams(text)) / den + 0.5 * float(pp) + (0.1 if sid == 0 else 0.0), int(pidx), sid, text, links))
    cands.sort(key=lambda c: -c[0])
    return cands[:POOL96_CAP]


def score_rr12(pairs, desc):
    """bge-reranker-v2-m3 的相關分數（單一 logit），fp16 推論。"""
    tok12 = AutoTokenizer.from_pretrained(RR12_MODEL)
    mdl12 = AutoModelForSequenceClassification.from_pretrained(RR12_MODEL, torch_dtype=torch.float16).to(device).eval()
    order = np.argsort([len(a) + len(b) for a, b in pairs])
    out = np.zeros(len(pairs), dtype=np.float32)
    with torch.no_grad():
        for i in tqdm(range(0, len(order), 64), desc=desc, mininterval=10):
            idx = order[i:i + 64]
            enc = tok12([pairs[j][0] for j in idx], [pairs[j][1] for j in idx], truncation="only_second", max_length=RR12_MAX_LEN,
                        padding=True, return_tensors="pt").to(device)
            out[idx] = mdl12(**enc).logits[:, 0].float().cpu().numpy()
    del mdl12
    free_gpu()
    return out


def pool96_scores(split):
    """回傳 (候選池, 原本重排器的分數, bge 分數)；分數有快取（bge 在 RTX 3050 上 dev 約 20 分鐘、test 約 40 分鐘）。"""
    rk = ranked[split["name"]]
    pools = [build_pool96(split["claim"][qi], rk[qi][0][:PAGE_TOPK], rk[qi][1][:PAGE_TOPK]) for qi in range(len(split["claim"]))]

    def build():
        pairs = [(split["claim"][qi], passage(c[1], c[3])) for qi in range(len(pools)) for c in pools[qi]]
        lg = score_pairs(reranker, pairs, RERANK_MAX_LEN, INFER_BS, f"原本的重排器 / 擴大候選池 {split['name']}")
        cut = np.cumsum([len(p) for p in pools])[:-1]
        return {"old": np.split((lg[:, 1] - lg[:, 0]).astype(np.float32), cut),
                "bge": np.split(score_rr12(pairs, f"bge-reranker / 擴大候選池 {split['name']}"), cut)}
    sc = cached(CACHE_DIR / f"pool96_{_sig}_{RR_TAG}_{split['name']}_{len(pools)}.pkl", build)
    return pools, sc["old"], sc["bge"]


P96 = {s["name"]: pool96_scores(s) for s in (dev, test)}
print("擴大候選池（dev）：平均每題", round(float(np.mean([len(p) for p in P96["dev"][0]])), 1), "句；標準證據組完整落在候選池的比例",
      round(evidence_recall(dev, [[(0.0, c[1], c[2]) for c in p] for p in P96["dev"][0]], POOL96_CAP), 4))
_d_old, _d_new = np.concatenate(P96["dev"][1]).astype(np.float64), np.concatenate(P96["dev"][2]).astype(np.float64)
_mo, _so, _mn, _sn = _d_old.mean(), _d_old.std(), _d_new.mean(), _d_new.std()


def v12_lists(name, w):
    """融合分數排序；每句為 (融合分數, 頁面, 句子編號, 文字, (), 原本重排器分數, bge 分數)。"""
    out = []
    for pl, s1, s2 in zip(*P96[name]):
        sc = (1 - w) * (np.asarray(s1, np.float64) - _mo) / _so + w * (np.asarray(s2, np.float64) - _mn) / _sn
        out.append([(float(v), c[1], c[2], c[3], (), float(a), float(b)) for v, c, a, b in sorted(zip(sc, pl, s1, s2), key=lambda t: -t[0])])
    return out


_V12_COLS = np.r_[np.arange(N_BASE), np.arange(N_BASE + N_LINK, N_BASE + N_LINK + N_RR2)]   # base 21 欄 + rr2 位置的 6 欄


def v12_s2(lists, which):     # which = 5（原本重排器的分數）或 6（bge 分數）
    return [np.array([c[which] for c in l[:SEL_N]], dtype=np.float32) for l in lists]


def v12_select(lists, which, test_lists=None):
    """與上方相同的選擇器：dev 為 5 折 OOF；test 用全部 dev 訓練的模型。"""
    X, meta = sel_features(dev, lists, ranked["dev"], v12_s2(lists, which))
    y = np.array([(titles[lists[qi][k][1]], lists[qi][k][2]) in _gold_dev[qi] for qi, k in meta])
    q = np.array([qi for qi, _ in meta])
    sr = np.array([dev["label"][qi] != 2 for qi in q])
    oof = np.zeros(len(y))
    for tr_i, va_i in GroupKFold(5).split(X, y, q):
        tr_i = tr_i[sr[tr_i]]
        oof[va_i] = sel_model().fit(X[tr_i][:, _V12_COLS], y[tr_i]).predict_proba(X[va_i][:, _V12_COLS])[:, 1]
    dev_l = sel_reorder(lists, meta, oof)
    if test_lists is None:
        return dev_l, None
    m_ = sel_model().fit(X[sr][:, _V12_COLS], y[sr])
    Xt, mt = sel_features(test, test_lists, ranked["test"], v12_s2(test_lists, which))
    return dev_l, sel_reorder(test_lists, mt, m_.predict_proba(Xt[:, _V12_COLS])[:, 1])


print("v12 證據句 Recall@1/3/5（dev，選擇器為 5 折 OOF）")
_v12_best = None
for _w in V12_W_GRID:
    _L = v12_lists("dev", _w)
    print(f"  融合 w = {_w:.2f}：只排序 {[round(evidence_recall(dev, _L, K), 4) for K in (1, 3, 5)]}")
    for _which in (5, 6):
        _dl, _ = v12_select(_L, _which)
        _r5 = evidence_recall(dev, _dl, 5)
        print(f"      + 選擇器（rr2 欄位放{'原本重排器' if _which == 5 else ' bge '}分數）：{[round(evidence_recall(dev, _dl, K), 4) for K in (1, 3, 5)]}")
        if _v12_best is None or _r5 > _v12_best[0]:
            _v12_best = (_r5, _w, _which)
_, V12_W, V12_WHICH = _v12_best
EV12 = dict(zip(("dev", "test"), v12_select(v12_lists("dev", V12_W), V12_WHICH, v12_lists("test", V12_W))))
print(f"採用 v12 證據：w = {V12_W}、rr2 欄位放{'原本重排器' if V12_WHICH == 5 else ' bge '}分數；dev Recall@1/3/5",
      [round(evidence_recall(dev, EV12["dev"], K), 4) for K in (1, 3, 5)], "（原本", [round(evidence_recall(dev, EV_OUT["dev"], K), 4) for K in (1, 3, 5)], "）")

證據句 Recall@1/3/5（dev，選擇器為 5 折 OOF）
  重排器 baseline         {1: 0.4955, 3: 0.8005, 5: 0.8615}
  選擇器 base           {1: 0.497, 3: 0.8275, 5: 0.878}
  選擇器 base+link      {1: 0.501, 3: 0.8285, 5: 0.878}
  選擇器 base+link+rr2  {1: 0.5105, 3: 0.8305, 5: 0.878}
採用證據句重新挑選： True | 特徵組合： base
  最短證據組 1 句（1273 題）Recall@5： 0.9356 → 0.9442
  最短證據組 2 句（513 題）Recall@5： 0.7856 → 0.8129
  最短證據組 3 句（169 題）Recall@5： 0.6509 → 0.6864


擴大候選池（dev）：平均每題 49.2 句；標準證據組完整落在候選池的比例 0.9245
v12 證據句 Recall@1/3/5（dev，選擇器為 5 折 OOF）
  融合 w = 0.00：只排序 [0.504, 0.813, 0.8725]


      + 選擇器（rr2 欄位放原本重排器分數）：[0.51, 0.844, 0.8905]


      + 選擇器（rr2 欄位放 bge 分數）：[0.5185, 0.8465, 0.891]
  融合 w = 0.50：只排序 [0.463, 0.8225, 0.8875]


      + 選擇器（rr2 欄位放原本重排器分數）：[0.518, 0.8455, 0.897]


      + 選擇器（rr2 欄位放 bge 分數）：[0.508, 0.8475, 0.899]
  融合 w = 0.75：只排序 [0.449, 0.8185, 0.8805]


      + 選擇器（rr2 欄位放原本重排器分數）：[0.52, 0.8465, 0.899]


      + 選擇器（rr2 欄位放 bge 分數）：[0.495, 0.847, 0.896]
  融合 w = 1.00：只排序 [0.41, 0.7545, 0.834]


      + 選擇器（rr2 欄位放原本重排器分數）：[0.517, 0.837, 0.8855]


      + 選擇器（rr2 欄位放 bge 分數）：[0.456, 0.82, 0.8765]


採用 v12 證據：w = 0.5、rr2 欄位放 bge 分數；dev Recall@1/3/5 [0.508, 0.8475, 0.899] （原本 [0.497, 0.8275, 0.878] ）


## 4. Classification：用 BERT 預測標籤

將 claim 與證據作為一組文字配對：`[CLS] claim [SEP] context [SEP]`。
tokenizer 把文字轉為 token ID、attention mask 等模型輸入。
模型的三個輸出分別對應 supports、refutes、NOT ENOUGH INFO。



In [13]:
def ctx_text(items):
    return " [SEP] ".join(passage(p, t) for p, _, t in items)


def build_cls_examples(seed=None):
    rg = random.Random(SEED if seed is None else seed)
    nei_ids = [i for i in range(len(train["claim"])) if train["label"][i] == 2]
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    if CLS_TRAIN_SR_CLAIMS:
        rg.shuffle(sr_ids)
        sr_ids = sorted(sr_ids[:CLS_TRAIN_SR_CLAIMS])

    def rerank_ids(ids, tag):
        sub = dict(claim=[train["claim"][i] for i in ids])
        first, final = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_train{tag}_{len(ids)}.pkl",
                              lambda: rerank_split(sub, [ranked["train"][i] for i in ids], f"重排 train-{tag}"))
        return [merge_hop(a, b, HOP_DELTA) for a, b in zip(first, final)]

    nei_lists, sr_lists = rerank_ids(nei_ids, "NEI"), rerank_ids(sr_ids, "SR")
    ex = []
    for pos_, qi in enumerate(tqdm(sr_ids, desc="建立分類訓練資料(S/R)", mininterval=5)):
        lst = sr_lists[pos_]
        groups = train["gold"][qi]
        best = min(groups, key=lambda g: (len({t for t, _ in g}), len(g)))
        by_key = {(titles[c[1]], c[2]): c for c in lst}
        top_score = lst[0][0] if lst else 0.0
        gold_items = []
        for t, s in best:
            if (t, s) in by_key:
                gold_items.append(by_key[(t, s)])
                continue
            j = title2idx.get(t)                       # 標準證據不在候選池：以最高分注入，確保這題看得到證據
            txt = next((x for sid, x, _ in (get_sentences(j) if j is not None else ()) if sid == s), None)
            if txt is None:
                gold_items = None
                break
            gold_items.append((top_score, j, s, txt, ()))
        if not gold_items:
            continue
        all_gold = {(t, s) for g in groups for t, s in g}
        fill = [c for c in lst if (titles[c[1]], c[2]) not in all_gold][:max(0, CLS_CTX_K - len(gold_items))]
        ctx = sorted(gold_items + fill, key=lambda c: -c[0])[:CLS_CTX_K]
        if rg.random() < CLS_SHUFFLE_P:
            rg.shuffle(ctx)
        ex.append((train["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in ctx]), int(train["label"][qi])))
    for pos_, qi in enumerate(nei_ids):
        ctx = list(nei_lists[pos_][:CLS_CTX_K])
        if ctx:
            if rg.random() < CLS_SHUFFLE_P:
                rg.shuffle(ctx)
            ex.append((train["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in ctx]), 2))
    rg.shuffle(ex)
    n_ctx = collections.Counter(e[1].count(" [SEP] ") + 1 for e in ex)
    print("每筆訓練資料的證據句數分佈（三類應該一致）：", dict(sorted(n_ctx.items())))
    return ex

### 模型與 Optimizer

`AutoModelForSequenceClassification` 會在預訓練 BERT 上加入三分類層。
分類層是新初始化的，因此載入時看到相關提示是正常的，需要用下面的資料訓練它。
傳入 `labels` 時，模型會自動計算 Cross Entropy loss。



In [14]:
_cls_path = CACHE_DIR / "classifier_v2"          # v2：修正訓練/測試證據分佈不一致的問題（v0 的 classifier 不再使用）
if _cls_path.exists() and not (FORCE_RETRAIN or FORCE_RETRAIN_CLASSIFIER):
    classifier = load_bundle(_cls_path)
    cls_prior = np.array(json.loads((_cls_path / "prior.json").read_text()))
    print("載入已訓練的分類器：", _cls_path)
else:
    cls_examples = build_cls_examples()
    offload(reranker)                      # 重排器之後不再需要，釋放顯存
    cls_prior = np.bincount([e[2] for e in cls_examples], minlength=3) / len(cls_examples)
    print("分類訓練資料：", len(cls_examples), "筆，類別比例 supports/refutes/NEI =", cls_prior.round(3))
    classifier = train_seq_clf(cls_examples, 3, CLS_MODEL, CLS_MAX_LEN, CLS_EPOCHS, CLS_LR, CLS_BS, CLS_ACCUM, "訓練分類器")
    save_bundle(classifier, _cls_path)
    (_cls_path / "prior.json").write_text(json.dumps(cls_prior.tolist()))
    del cls_examples
offload(classifier)                        # 之後每個成員都是「用到才從硬碟載入」，一次只佔一個模型的顯存
free_gpu()


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading weights:   7%|▋         | 14/201 [00:00<00:01, 133.90it/s]


Loading weights:  14%|█▍        | 28/201 [00:00<00:01, 126.00it/s]


Loading weights:  22%|██▏       | 44/201 [00:00<00:01, 132.08it/s]


Loading weights:  38%|███▊      | 76/201 [00:00<00:00, 200.86it/s]


Loading weights:  58%|█████▊    | 117/201 [00:00<00:00, 264.81it/s]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 339.15it/s]

載入已訓練的分類器： output/cache/classifier_v2


### Training

每批資料依序執行：清空梯度、計算預測與 loss、反向傳播、更新參數。
`model.train()` 會啟用訓練模式；gradient clipping 用來限制過大的梯度。



In [15]:
dev_y = dev["label"]


# ----- 分類器集成：成員訓練、逐成員推論（logits 有快取）-----
offload(reranker)                          # 重排器之後不再使用（訓練資料的重排結果都已快取）
free_gpu()
# 先看 claim + 前 4 句證據的 token 數：若常超過 256，會把後面的證據截掉（新成員會自動改用 384）
try:
    _tk = AutoTokenizer.from_pretrained(CLS_MODEL)
    _tk.model_max_length = 100000
    _dp = [(dev["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in EV["dev"][qi][:CLS_CTX_K]])) for qi in range(len(dev["claim"]))]
    _ln = np.array([len(x) for x in _tk([a for a, _ in _dp], [b for _, b in _dp], truncation=False)["input_ids"]])
    TRUNC_RATE = float((_ln > CLS_MAX_LEN).mean())
    print(f"dev 的輸入長度（token）：中位數 {np.median(_ln):.0f}、90 百分位 {np.percentile(_ln, 90):.0f}；超過 {CLS_MAX_LEN} 而被截斷的比例 = {TRUNC_RATE:.3f}")
except Exception as e:
    print("（略過輸入長度檢查：", repr(e)[:80], "）")
    TRUNC_RATE = 0.0
EXT_LEN = 384 if TRUNC_RATE > 0.10 else CLS_MAX_LEN        # 新成員使用的最大長度
OOM_ERR = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)


def member_path(m):
    return CACHE_DIR / (m.get("path") or f"cls_{m['tag']}")


def train_member(m):
    path = member_path(m)
    if path.exists() and not (FORCE_RETRAIN or FORCE_RETRAIN_CLASSIFIER):
        return
    ex = build_cls_examples(seed=m.get("seed"))
    if m.get("kind") == "llm":
        if m.get("n_train"):
            ex = ex[:m["n_train"]]
        print(f"  {m['tag']}：{len(ex)} 筆訓練資料；留意進度條的速度，可估算全部訓練時間")
        bundle = train_llm_clf(ex, m["model"], m.get("max_len", 384), m.get("epochs", 2), m.get("lr", 1e-4), m.get("bs", 4),
                               m.get("accum", 4), f"訓練 {m['tag']}", seed=m.get("seed"))
        save_llm(bundle, path)
        (path / "meta.json").write_text(json.dumps(dict(kind="llm", max_len=m.get("max_len", 384), base=m["model"])))
        offload(bundle); del bundle, ex
        free_gpu()
        return
    if m.get("swap"):
        ex = [(b, a, y) for a, b, y in ex]                    # (證據, claim)：NLI 的 前提/假設 順序
    freeze, max_len = m.get("freeze", 0), m.get("max_len", EXT_LEN)
    while True:
        oom, bundle = False, None
        try:
            bundle = train_seq_clf(ex, 3, m["model"], max_len, m.get("epochs", CLS_EPOCHS), m.get("lr", CLS_LR), m.get("bs", CLS_BS),
                                   m.get("accum", CLS_ACCUM), f"訓練 {m['tag']}", seed=m.get("seed"), freeze=freeze, optim=m.get("optim", "adamw"), head_perm=m.get("head_perm"))
        except OOM_ERR:
            oom = True
        if not oom:
            break
        gc.collect(); free_gpu()
        freeze += 4
        if freeze > 16:
            raise RuntimeError(f"{m['tag']}：顯存不足，已凍結到第 16 層仍然放不下；請改用較小的模型或把 bs 降為 2")
        print(f"  顯存不足 → 改為凍結前 {freeze} 層後重試")
    save_bundle(bundle, path)
    (path / "meta.json").write_text(json.dumps(dict(max_len=max_len, freeze=freeze, swap=bool(m.get("swap")))))
    offload(bundle); del bundle, ex
    free_gpu()


for _m in CLS_MEMBERS:
    if _m["tag"] in ACTIVE_MEMBERS:
        train_member(_m)


def member_logits(m, split, lists, key=""):
    path = member_path(m)
    meta = json.loads((path / "meta.json").read_text()) if (path / "meta.json").exists() else {}
    max_len = meta.get("max_len", m.get("max_len", CLS_MAX_LEN))
    cache = None
    if split["name"] in ("dev", "test"):
        _cf = path / "config.json" if (path / "config.json").exists() else path / "adapter_config.json"   # LLM 只存 LoRA 轉接器
        cache = CACHE_DIR / f"cls_logits_{m['tag']}_{int(_cf.stat().st_mtime)}_{RR_TAG}_{HOP_DELTA}_{split['name']}{key}.npy"
        if cache.exists():
            return np.load(cache)
    pairs = [(split["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in lists[qi][:CLS_CTX_K]])) for qi in range(len(split["claim"]))]
    if m.get("kind") == "llm":
        bundle = load_llm(path, m["model"])
        lg = score_llm(bundle, [llm_parts(a, b) for a, b in pairs], max_len, m.get("infer_bs", 16), f"分類 {split['name']} / {m['tag']}")
        offload(bundle); del bundle
        free_gpu()
        if cache is not None:
            np.save(cache, lg)
        return lg
    if meta.get("swap"):
        pairs = [(b, a) for a, b in pairs]
    bundle = load_bundle(path)
    lg = score_pairs(bundle, pairs, max_len, m.get("infer_bs", INFER_BS), f"分類 {split['name']} / {m['tag']}")
    offload(bundle); del bundle
    free_gpu()
    if cache is not None:
        np.save(cache, lg)
    return lg


def log_softmax(x):
    x = x - x.max(1, keepdims=True)
    return x - np.log(np.exp(x).sum(1, keepdims=True))


# ===== 9b. 本機訓練的新成員：完整版 Qwen3-0.6B、附兄弟 claim 的 Qwen3-0.6B =====
# qwen06（設定在第 1 節的 CLS_MEMBERS）：全部訓練資料、2 epoch，RTX 3050 約 4.5 小時；訓練完會存檔，之後重跑直接載入。
QWEN_FULL = next(m for m in CLS_MEMBERS if m["tag"] == "qwen06")
train_member(QWEN_FULL)
SIB_N, SIB_GAP_ID, SIB_MIN_JAC = 2, 20, 0.10
QWEN_SIB_LEN, QWEN_SIB_SEED = 448, 41


def sibling_claims(split, n=SIB_N):
    """同一切分中 id 相差 <= 20、字元二元組 Jaccard >= 0.1 的 claim，依 Jaccard 由高到低取 n 句（不使用標籤）。"""
    ids = np.array(split["id"]); order = np.argsort(ids, kind="stable")
    B = [_bg(c) for c in split["claim"]]
    out = [[] for _ in ids]
    for pos, qi in enumerate(order):
        cand = []
        for pj in range(max(0, pos - 12), min(len(order), pos + 13)):
            qj = order[pj]
            if qj != qi and abs(int(ids[qj]) - int(ids[qi])) <= SIB_GAP_ID:
                jac = len(B[qi] & B[qj]) / max(1, len(B[qi] | B[qj]))
                if jac >= SIB_MIN_JAC:
                    cand.append((jac, qj))
        cand.sort(key=lambda t: (-t[0], t[1]))
        out[qi] = [split["claim"][qj] for _, qj in cand[:n]]
    return out


def with_sibs(ctx, sibs):
    return ctx + (" [SEP] 相關敘述：" + "；".join(sibs) if sibs else "")


SIBS = {s["name"]: sibling_claims(s) for s in (train, dev, test)}
print("有兄弟 claim 的比例：", {k: round(float(np.mean([len(x) > 0 for x in v])), 3) for k, v in SIBS.items()})


def qwen_sib_parts(split):
    return [llm_parts(split["claim"][qi], with_sibs(ctx_text([(c[1], c[2], c[3]) for c in EV[split["name"]][qi][:CLS_CTX_K]]), SIBS[split["name"]][qi]))
            for qi in range(len(split["claim"]))]


def qwen_sib_member():
    """附兄弟 claim 的 Qwen3-0.6B + LoRA（只用 train 訓練）；回傳 {(split, "rr"): log-softmax}。"""
    path = CACHE_DIR / "cls_qwen06_sib"
    files = {s: CACHE_DIR / f"cls_logits_qwen06_sib_{s}.npy" for s in ("dev", "test")}
    if not all(p.exists() for p in files.values()):
        if not path.exists():
            first_sibs = {}
            for qi, c in enumerate(train["claim"]):
                first_sibs.setdefault(c, SIBS["train"][qi])
            ex = [(c, with_sibs(ctx, first_sibs.get(c, [])), lab) for c, ctx, lab in build_cls_examples(seed=QWEN_SIB_SEED)]
            bundle = train_llm_clf(ex, "Qwen/Qwen3-0.6B-Base", QWEN_SIB_LEN, 2, 1e-4, 4, 4, "訓練 qwen06_sib", seed=QWEN_SIB_SEED)
            save_llm(bundle, path); offload(bundle); del bundle, ex
            free_gpu()
        bundle = load_llm(path, "Qwen/Qwen3-0.6B-Base")
        for s in (dev, test):
            np.save(files[s["name"]], score_llm(bundle, qwen_sib_parts(s), QWEN_SIB_LEN, 16, f"分類 {s['name']} / qwen06_sib"))
        offload(bundle); del bundle
        free_gpu()
    return {(s, "rr"): log_softmax(np.load(files[s])) for s in ("dev", "test")}


LOCAL9 = {"qwen06": {(s["name"], "rr"): log_softmax(member_logits(QWEN_FULL, s, EV[s["name"]])) for s in (dev, test)},
          "qwen06_sib": qwen_sib_member()}
for _t, _r in LOCAL9.items():
    print(f"本機新成員 {_t}：dev accuracy {(_r[('dev', 'rr')].argmax(1) == dev_y).mean():.4f}")


# ===== 9d. 在 Kaggle Notebook GPU（16GB）上訓練的成員 =====
# 本機 RTX 3050 只有 4GB，330M 模型必須凍結前 8 層、輸入只能放 4 句；下列成員改在 Kaggle 的 GPU（作業說明提到每週 30 小時）訓練：
#   - 完整微調（不凍結）、最大長度 512；不同模型家族（XLM-R-large-XNLI、mDeBERTa-v3-XNLI）；多數成員在輸入中附上 2～3 句「兄弟 claim」（與 9b 相同的定義，不用標籤）。
#   - 訓練資料與推論輸入由 export_for_kaggle() 從本 notebook 的重排結果匯出（與 build_cls_examples 的建構方式相同），
#     訓練程式碼就是下面的 KG_SCRIPT（上傳成 Kaggle 私人 dataset + 私人 notebook 執行），輸出 dev / test 兩種視角的 logits，下載到 output/cache/kaggle/<tag>/。
#   - 本機重跑時直接讀取下載的 logits；若本機有 >= 15GB 顯存的 GPU，也可以把 KG_RUN_LOCAL 設成 True 直接在本機訓練。
KG_DIR = CACHE_DIR / "kaggle"
KG_DATA = KG_DIR / "data"
KG_RUN_LOCAL = False
KG_SPECS = [
    dict(tag="xlmr_xnli_k6", model="joeddav/xlm-roberta-large-xnli", K=6, max_len=512, lr=8e-6, epochs=2, bs=8, accum=2, seed=11,
         swap=True, head_perm=[2, 0, 1], cf=2),
    dict(tag="e330_sib2_k4", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", K=4, sib=2, max_len=512, lr=1e-5, epochs=2, bs=16, accum=1, seed=13,
         swap=True, head_perm=[2, 0, 1], cf=0),
    dict(tag="xlmr_sib2_k4", model="joeddav/xlm-roberta-large-xnli", K=4, sib=2, max_len=512, lr=8e-6, epochs=2, bs=8, accum=2, seed=19,
         swap=True, head_perm=[2, 0, 1], cf=0),
    dict(tag="e330_sib3_k6", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", K=6, sib=3, max_len=512, lr=1e-5, epochs=2, bs=16, accum=1, seed=17,
         swap=True, head_perm=[2, 0, 1], cf=0),
    dict(tag="mdeberta_sib2_k4", model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7", K=4, sib=2, max_len=512, lr=2e-5, epochs=2, bs=16,
         accum=1, seed=23, swap=True, head_perm=[0, 2, 1], cf=0),
]


def export_for_kaggle(topn=10):
    """匯出 train / dev / test 的 claim、重排器前 topn 句（含分數）、選擇器前 topn 句、標準證據句（含文字）。"""
    KG_DATA.mkdir(parents=True, exist_ok=True)
    L = lambda lst: [[disp(c[1]), int(c[2]), round(float(c[0]), 4), c[3], titles[c[1]]] for c in lst[:topn]]

    def gold_groups(groups):
        out = []
        for gr in groups:
            items = []
            for t, s in gr:
                j = title2idx.get(t)
                txt = next((x for sid, x, _ in (get_sentences(j) if j is not None else ()) if sid == s), None)
                items.append([disp(j) if j is not None else t, int(s), txt, t])
            out.append(items)
        return out
    lists = {}
    nei_ids = [i for i in range(len(train["claim"])) if train["label"][i] == 2]
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    for ids, tag in ((nei_ids, "NEI"), (sr_ids, "SR")):
        first, _ = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_train{tag}_{len(ids)}.pkl", lambda: None)
        lists.update(zip(ids, first))
    with open(KG_DATA / "train.jsonl", "w", encoding="utf-8") as f:
        for qi in range(len(train["claim"])):
            f.write(json.dumps(dict(id=int(train["id"][qi]), claim=train["claim"][qi], label=int(train["label"][qi]), rr=L(lists[qi]),
                                    gold=gold_groups(train["gold"][qi])), ensure_ascii=False) + "\n")
    with open(KG_DATA / "dev.jsonl", "w", encoding="utf-8") as f:
        for qi in range(len(dev["claim"])):
            f.write(json.dumps(dict(id=int(dev["id"][qi]), claim=dev["claim"][qi], label=int(dev["label"][qi]), rr=L(rr["dev"][0][qi]),
                                    sel=L(EV_OUT["dev"][qi]), gold=gold_groups(dev["gold"][qi])), ensure_ascii=False) + "\n")
    with open(KG_DATA / "test.jsonl", "w", encoding="utf-8") as f:
        for qi in range(len(test["claim"])):
            f.write(json.dumps(dict(id=int(test["id"][qi]), claim=test["claim"][qi], rr=L(rr["test"][0][qi]), sel=L(EV_OUT["test"][qi])),
                               ensure_ascii=False) + "\n")


KG_SCRIPT = r'''# CFEVER classifier member training on a Kaggle GPU (same example construction as build_cls_examples in the notebook).
# Generative AI usage: written with the help of Claude (Anthropic); see the notebook's GenAI statement.
import os, json, math, random, time, glob
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

CFG = __CFG__
print("CFG", CFG, flush=True)
DATA = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
OUT = "/kaggle/working"
dev_ = torch.device("cuda")
rd = lambda n: [json.loads(l) for l in open(f"{DATA}/{n}.jsonl", encoding="utf-8")]
train, dev, test = rd("train"), rd("dev"), rd("test")
K, P_SHUF = CFG["K"], 0.3


def add_siblings(rows, n):
    """Attach up to n most similar id-adjacent claims of the same split (no labels used)."""
    ids = np.array([r["id"] for r in rows]); order = np.argsort(ids, kind="stable")
    B = [{c[i:i + 2] for i in range(len(c) - 1)} for c in (r["claim"] for r in rows)]
    for pos, qi in enumerate(order):
        cand = []
        for pj in range(max(0, pos - 12), min(len(order), pos + 13)):
            qj = order[pj]
            if qj != qi and abs(int(ids[qj]) - int(ids[qi])) <= 20:
                jac = len(B[qi] & B[qj]) / max(1, len(B[qi] | B[qj]))
                if jac >= 0.1:
                    cand.append((jac, qj))
        cand.sort(key=lambda t: (-t[0], t[1]))
        rows[qi]["sibs"] = [rows[qj]["claim"] for _, qj in cand[:n]]


NSIB = CFG.get("sib", 0)
if NSIB:
    for rows_ in (train, dev, test):
        add_siblings(rows_, NSIB)


def with_sibs(r, ctx):
    if not NSIB or not r.get("sibs"):
        return ctx
    return ctx + " [SEP] 相關敘述：" + "；".join(r["sibs"])


def ctx_text(items):
    return " [SEP] ".join(f"{c[0]}：{c[3]}" for c in items)


def examples(rows, seed, order_sr_first=True):
    """rows: dicts with claim/label/rr/gold. S/R: inject gold (best group) into top-K; NEI: retrieved top-K. 30% shuffle."""
    rg = random.Random(seed)
    ex = []
    sr = [r for r in rows if r["label"] != 2]
    nei = [r for r in rows if r["label"] == 2]
    for r in sr:
        lst, groups = r["rr"], r["gold"]
        best = min(groups, key=lambda g: (len({it[3] for it in g}), len(g)))
        by_key = {(c[4], c[1]): c for c in lst}
        top = lst[0][2] if lst else 0.0
        items = []
        for it in best:
            if (it[3], it[1]) in by_key:
                items.append(by_key[(it[3], it[1])]); continue
            if it[2] is None:
                items = None; break
            items.append([it[0], it[1], top, it[2], it[3]])
        if not items:
            continue
        allg = {(it[3], it[1]) for g in groups for it in g}
        fill = [c for c in lst if (c[4], c[1]) not in allg][:max(0, K - len(items))]
        ctx = sorted(items + fill, key=lambda c: -c[2])[:K]
        if rg.random() < P_SHUF:
            rg.shuffle(ctx)
        ex.append((r["claim"], with_sibs(r, ctx_text(ctx)), r["label"]))
    for r in nei:
        ctx = list(r["rr"][:K])
        if ctx:
            if rg.random() < P_SHUF:
                rg.shuffle(ctx)
            ex.append((r["claim"], with_sibs(r, ctx_text(ctx)), 2))
    rg.shuffle(ex)
    return ex


def pairs(rows, view):
    return [(r["claim"], with_sibs(r, ctx_text(r[view][:K]))) for r in rows]


def train_enc(ex, seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    tok = AutoTokenizer.from_pretrained(CFG["model"])
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model"], num_labels=3, torch_dtype=torch.float32).to(dev_)
    hp = CFG.get("head_perm")
    if hp:                                              # reorder the NLI head (contradiction/neutral/entailment -> supports/refutes/NEI)
        head = model.classifier.out_proj if hasattr(model.classifier, "out_proj") else model.classifier
        with torch.no_grad():
            pm = torch.tensor(hp, device=dev_)
            head.weight.copy_(head.weight[pm].clone()); head.bias.copy_(head.bias[pm].clone())
    if CFG.get("grad_ckpt", True):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=0.01)
    bs, acc, ep = CFG["bs"], CFG["accum"], CFG["epochs"]
    steps = math.ceil(len(ex) / (bs * acc)) * ep
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    rg = random.Random(seed)
    swap = CFG.get("swap", False)
    t0 = time.time()
    for e in range(ep):
        model.train(); order = list(range(len(ex))); rg.shuffle(order); pend = 0; run = 0.0
        for bi, s in enumerate(range(0, len(order), bs)):
            b = [ex[i] for i in order[s:s + bs]]
            a1, a2 = [x[0] for x in b], [x[1] for x in b]
            enc = tok(a2 if swap else a1, a1 if swap else a2, truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt").to(dev_)
            lab = torch.tensor([x[2] for x in b], device=dev_)
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(**enc).logits
            loss = F.cross_entropy(lg.float(), lab, label_smoothing=CFG.get("ls", 0.0)) / acc
            scaler.scale(loss).backward(); pend += 1
            run = 0.98 * run + 0.02 * loss.item() * acc if bi else loss.item() * acc
            if pend == acc or s + bs >= len(order):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True); pend = 0
            if bi % 200 == 0:
                print(f"ep {e} step {bi}/{len(order)//bs} loss {run:.4f} {time.time()-t0:.0f}s", flush=True)
    model.eval()
    return model, tok


@torch.no_grad()
def infer(model, tok, prs):
    swap = CFG.get("swap", False)
    order = np.argsort([len(a) + len(b) for a, b in prs], kind="stable"); out = np.zeros((len(prs), 3), np.float32)
    for s in range(0, len(order), 64):
        ids = order[s:s + 64]
        a1, a2 = [prs[i][0] for i in ids], [prs[i][1] for i in ids]
        enc = tok(a2 if swap else a1, a1 if swap else a2, truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt").to(dev_)
        with torch.autocast("cuda", dtype=torch.float16):
            out[ids] = model(**enc).logits.float().cpu().numpy()
    return out


tag = CFG["tag"]
nf = CFG.get("cf", 0)                                   # 0 = train only; 2 = cross-fit with dev halves (contiguous ids)
dev_order = np.argsort([r["id"] for r in dev], kind="stable")
folds = np.array_split(dev_order, nf) if nf else [np.array([], dtype=int)]
base_ex = examples(train, CFG["seed"])
for f, held in enumerate(folds):
    held = set(held.tolist())
    ex = list(base_ex)
    if nf:
        extra = examples([dev[q] for q in range(len(dev)) if q not in held], seed=1000 + f)
        ex = ex + extra; random.Random(77 + f).shuffle(ex)
    print(f"fold {f}: {len(ex)} examples", flush=True)
    model, tok = train_enc(ex, CFG["seed"] + f)
    for split, rows in (("dev", dev), ("test", test)):
        for view in ("rr", "sel"):
            lg = infer(model, tok, pairs(rows, view))
            np.save(f"{OUT}/{tag}_f{f}_{split}_{view}.npy", lg)
            if split == "dev":
                yy = np.array([r["label"] for r in dev]); idx = np.array(sorted(held)) if nf else np.arange(len(dev))
                print(f"fold {f} {view} dev acc (held-out part) {(lg[idx].argmax(1) == yy[idx]).mean():.4f}", flush=True)
    del model; torch.cuda.empty_cache()
print("DONE", flush=True)
'''


def kaggle_views(spec):
    """讀取（或在本機大顯存 GPU 上重新訓練）一個 Kaggle 成員，回傳 {(split, view): log-softmax}；dev 為 OOF（cf=0 時為只用 train 訓練的模型）。"""
    d = KG_DIR / spec["tag"]
    nf = spec.get("cf", 0)
    need = [d / f"{spec['tag']}_f{f}_{s}_{v}.npy" for f in range(max(nf, 1)) for s in ("dev", "test") for v in ("rr", "sel")]
    if not all(p.exists() for p in need):
        if not KG_RUN_LOCAL:
            print(f"  （略過 {spec['tag']}：找不到 {d} 的 logits；請在 Kaggle GPU 上執行 KG_SCRIPT，或設 KG_RUN_LOCAL=True）")
            return None
        if not (KG_DATA / "train.jsonl").exists():
            export_for_kaggle()
        d.mkdir(parents=True, exist_ok=True)
        code = KG_SCRIPT.replace("__CFG__", repr(spec)).replace('glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]', repr(str(KG_DATA / "train.jsonl")))
        exec(compile(code.replace('OUT = "/kaggle/working"', f"OUT = {str(d)!r}"), "kaggle_member", "exec"), {"__name__": "kaggle_member"})
    ids = np.array(dev["id"])
    folds = np.array_split(np.argsort(ids, kind="stable"), nf) if nf else [np.arange(len(ids))]
    res = {}
    for v in ("rr", "sel"):
        oof = np.zeros((len(ids), 3), np.float32)
        tests = []
        for f in range(max(nf, 1)):
            oof[folds[f]] = log_softmax(np.load(d / f"{spec['tag']}_f{f}_dev_{v}.npy"))[folds[f]]
            tests.append(log_softmax(np.load(d / f"{spec['tag']}_f{f}_test_{v}.npy")))
        res[("dev", v)], res[("test", v)] = oof, np.mean(tests, axis=0)
    return res


KG = {}
for _s in KG_SPECS:
    _r = kaggle_views(_s)
    if _r is not None:
        KG[_s["tag"]] = _r
        print(f"Kaggle 成員 {_s['tag']}：dev accuracy 重排器視角 {(_r[('dev', 'rr')].argmax(1) == dev_y).mean():.4f}、選擇器視角 {(_r[('dev', 'sel')].argmax(1) == dev_y).mean():.4f}"
              f"（{'OOF，cross-fit' if _s.get('cf') else '只用 train 訓練'}）")

dev 的輸入長度（token）：中位數 253、90 百分位 357；超過 256 而被截斷的比例 = 0.485


有兄弟 claim 的比例： {'train': 0.996, 'dev': 0.995, 'test': 0.726}
本機新成員 qwen06：dev accuracy 0.7630
本機新成員 qwen06_sib：dev accuracy 0.7753
Kaggle 成員 xlmr_xnli_k6：dev accuracy 重排器視角 0.7707、選擇器視角 0.7740（OOF，cross-fit）
Kaggle 成員 e330_sib2_k4：dev accuracy 重排器視角 0.7697、選擇器視角 0.7790（只用 train 訓練）
Kaggle 成員 xlmr_sib2_k4：dev accuracy 重排器視角 0.7657、選擇器視角 0.7673（只用 train 訓練）
Kaggle 成員 e330_sib3_k6：dev accuracy 重排器視角 0.7720、選擇器視角 0.7740（只用 train 訓練）
Kaggle 成員 mdeberta_sib2_k4：dev accuracy 重排器視角 0.7553、選擇器視角 0.7587（只用 train 訓練）


### Prediction

`model.eval()` 關閉訓練模式，`torch.no_grad()` 表示不計算梯度。
`argmax` 選出分數最大的類別編號，再轉回標籤文字。



In [16]:
MEMBERS = [m for m in CLS_MEMBERS if m["tag"] in ACTIVE_MEMBERS]


def classify(split, final_lists, members=None):
    """集成：各成員 log-softmax 的平均（之後的偏差校準直接加在這個分數上）。"""
    ms = members or MEMBERS
    return np.mean([log_softmax(member_logits(m, split, final_lists)) for m in ms], axis=0)


def predict_with_prior(logits, correct):
    if correct:
        logits = logits + np.log(np.array(PRIOR_TARGET) / np.maximum(cls_prior, 1e-6))
    return logits.argmax(1)


dev_logits = classify(dev, EV["dev"])
dev_y = dev["label"]
dev_ev_ok = np.array([lab == 2 or any(all(x in {(titles[c[1]], c[2]) for c in EV_OUT["dev"][qi][:5]} for x in g) for g in dev["gold"][qi])
                      for qi, lab in enumerate(dev_y)])      # NEI 不看證據；S/R 需要前 5 句涵蓋任一組標準證據


test_logits = classify(test, EV["test"])


def nei_share(lg, b):
    return float(((lg + np.array([0, 0, b])).argmax(1) == 2).mean())


def bias_for_share(lg, share):
    lo, hi = -3.0, 8.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if nei_share(lg, mid) < share else (lo, mid)
    return (lo + hi) / 2


DEV_SHARES = (0.42, 0.46, 0.50)                                    # 部署時 dev 的 NEI 預測比例大約落在這個範圍


def paired_at_shares(lg_a, lg_b, ok=dev_ev_ok):
    """在相同的 dev NEI 預測比例下比較兩個集成（同一批 dev 題目的配對檢定）。"""
    out = []
    for s_ in DEV_SHARES:
        a_ = ((lg_a + np.array([0, 0, bias_for_share(lg_a, s_)])).argmax(1) == dev_y) & ok
        b_ = ((lg_b + np.array([0, 0, bias_for_share(lg_b, s_)])).argmax(1) == dev_y) & ok
        n10, n01 = int((~a_ & b_).sum()), int((a_ & ~b_).sum())
        out.append((b_.mean() - a_.mean(), n10, n01, (n10 - n01) / math.sqrt(max(1, n10 + n01))))
    return out


ALL9 = {m["tag"]: {(s["name"], "rr"): log_softmax(member_logits(m, s, EV[s["name"]])) for s in (dev, test)} for m in MEMBERS}
ALL9.update(LOCAL9)
ALL9.update(KG)
V7_TAGS = [m["tag"] for m in MEMBERS]
STEPS = [("v7（6 成員）", V7_TAGS),
         ("pilot → 完整版 qwen06", [t for t in V7_TAGS if t != "qwen06_pilot"] + ["qwen06"])]
for _t in ["xlmr_xnli_k6", "e330_sib2_k4", "xlmr_sib2_k4", "e330_sib3_k6", "mdeberta_sib2_k4", "qwen06_sib"]:
    if _t in ALL9:
        STEPS.append((f"+ {_t}", STEPS[-1][1] + [_t]))
ens9 = lambda tags, split: np.mean([ALL9[t][(split, "rr")] for t in tags], axis=0)
_v7d = ens9(V7_TAGS, "dev")


V9_TAGS = STEPS[-1][1]                                             # 最終集成：全部步驟的成員
dev_logits9, test_logits9 = ens9(V9_TAGS, "dev"), ens9(V9_TAGS, "test")


# ===== v12：本機的 7 個成員改讀 v12 證據的前 4 句（權重不變，只重新推論；見 Report Q3）=====
# Kaggle 上訓練的 5 個成員只保存了 logits（權重沒有下載），所以維持原本的證據；集成仍是 12 個成員的 log-softmax 平均。
def qwen_sib_parts_v12(split):
    return [llm_parts(split["claim"][qi], with_sibs(ctx_text([(c[1], c[2], c[3]) for c in EV12[split["name"]][qi][:CLS_CTX_K]]), SIBS[split["name"]][qi]))
            for qi in range(len(split["claim"]))]


def qwen_sib_v12():
    files = {s: CACHE_DIR / f"cls_logits_qwen06_sib_{s}_v12.npy" for s in ("dev", "test")}
    if not all(p.exists() for p in files.values()):
        bundle = load_llm(CACHE_DIR / "cls_qwen06_sib", "Qwen/Qwen3-0.6B-Base")
        for s in (dev, test):
            np.save(files[s["name"]], score_llm(bundle, qwen_sib_parts_v12(s), QWEN_SIB_LEN, 16, f"分類 {s['name']} / qwen06_sib（v12 證據）"))
        offload(bundle); del bundle
        free_gpu()
    return {(s, "rr"): log_softmax(np.load(files[s])) for s in ("dev", "test")}


V12_LOCAL = {m["tag"]: {(s["name"], "rr"): log_softmax(member_logits(m, s, EV12[s["name"]], key="_v12")) for s in (dev, test)}
             for m in [m for m in MEMBERS if m["tag"] in V9_TAGS] + [QWEN_FULL]}
V12_LOCAL["qwen06_sib"] = qwen_sib_v12()
ALL12 = {t: V12_LOCAL.get(t, ALL9[t]) for t in V9_TAGS}
dev_logits12 = np.mean([ALL12[t][("dev", "rr")] for t in V9_TAGS], axis=0)
test_logits12 = np.mean([ALL12[t][("test", "rr")] for t in V9_TAGS], axis=0)
for _t in V12_LOCAL:
    print(f"v12 本機成員 {_t}：dev accuracy {(ALL9[_t][('dev', 'rr')].argmax(1) == dev_y).mean():.4f} → {(ALL12[_t][('dev', 'rr')].argmax(1) == dev_y).mean():.4f}")
print(f"v12 集成（12 成員）dev accuracy：{(dev_logits9.argmax(1) == dev_y).mean():.4f} → {(dev_logits12.argmax(1) == dev_y).mean():.4f}")

v12 本機成員 macbert_base：dev accuracy 0.7137 → 0.7203
v12 本機成員 roberta_base：dev accuracy 0.7227 → 0.7210
v12 本機成員 erlangshen_110m：dev accuracy 0.7470 → 0.7580
v12 本機成員 erlangshen_330m：dev accuracy 0.7663 → 0.7667
v12 本機成員 erlangshen_330m_b：dev accuracy 0.7647 → 0.7600
v12 本機成員 qwen06：dev accuracy 0.7630 → 0.7720
v12 本機成員 qwen06_sib：dev accuracy 0.7753 → 0.7820
v12 集成（12 成員）dev accuracy：0.7963 → 0.8037


### Classification Accuracy

只比對分類標籤，不要求證據正確；所有題目（包含 NEI）都進入分母。
因此可以單獨觀察 BERT 分類器的表現，不會與檢索分數混在一起。



In [17]:
def cfever_score(split, pred, lists):
    hit = 0
    for qi in range(len(split["claim"])):
        if pred[qi] != split["label"][qi]:
            continue
        if split["label"][qi] == 2:
            hit += 1
            continue
        top = {(titles[c[1]], c[2]) for c in lists[qi][:5]}
        hit += any(all(x in top for x in g) for g in split["gold"][qi])
    return hit / len(split["claim"])


def cf(pred, y, ok):
    return float(((pred == y) & ok).mean())


member_acc = {m["tag"]: float((member_logits(m, dev, EV["dev"]).argmax(1) == dev_y).mean()) for m in MEMBERS}
print("各成員 dev accuracy（未校準）：", {k: round(v, 4) for k, v in member_acc.items()}, "| 集成：", round(float((dev_logits.argmax(1) == dev_y).mean()), 4))
acc_plain = float((dev_logits.argmax(1) == dev_y).mean())
acc_prior = float((predict_with_prior(dev_logits, True) == dev_y).mean())   # 舊做法（類別先驗修正），僅作消融
cf_plain = cf(dev_logits.argmax(1), dev_y, dev_ev_ok)
# 集成成員的所有組合（未校準）：看哪些成員真的有貢獻
import itertools
_ml = {m["tag"]: log_softmax(member_logits(m, dev, EV["dev"])) for m in MEMBERS}
subset_rows = []
for r_ in range(1, len(MEMBERS) + 1):
    for comb in itertools.combinations(_ml, r_):
        lg = np.mean([_ml[t] for t in comb], axis=0)
        subset_rows.append(dict(members="+".join(comb), acc=float((lg.argmax(1) == dev_y).mean()), cfever=cf(lg.argmax(1), dev_y, dev_ev_ok)))
subset_df = pd.DataFrame(subset_rows).sort_values("cfever", ascending=False).reset_index(drop=True)
print("成員組合的 dev 表現（未校準）：")
print(subset_df.round(4).to_string(index=False))

# ----- 新成員的配對比較（McNemar）：同一批 dev 題目，看加入新成員前後各翻轉了幾題 -----
# 只看「平均分數差多少」會被雜訊誤導：兩個相近的集成通常有幾百題預測不同，差異的標準誤大約是 sqrt(翻轉題數)/題數。
# 這裡直接印出兩個方向的翻轉題數，差值小於 2 個標準誤就代表「分不出高下」，不該據此換模型。
if NEW_MEMBER in _ml and len(_ml) > 1:
    def _hit(tags):
        p = np.mean([_ml[t] for t in tags], axis=0).argmax(1)
        return (p == dev_y) & dev_ev_ok
    _wo = _hit([t for t in _ml if t != NEW_MEMBER])
    _wi = _hit(list(_ml))
    _n01, _n10 = int((_wo & ~_wi).sum()), int((~_wo & _wi).sum())
    _d = (_n10 - _n01) / len(dev_y)
    _se = math.sqrt(max(_n01 + _n10, 1)) / len(dev_y)
    print(f"\n加入 {NEW_MEMBER} 的配對比較（dev CFEVER）：{_wo.mean():.4f} → {_wi.mean():.4f}（差 {_d:+.4f}）")
    print(f"  變好 {_n10} 題、變壞 {_n01} 題 → 標準誤約 {_se:.4f}；差值 / 標準誤 = {_d / _se:+.2f}")
    print("  |差值| < 2 倍標準誤 = 統計上分不出高下；此時留著新成員也無妨（平均通常不會變差），但別只憑這個數字換掉已確認分數的提交檔。")

# ----- 校準：替 refutes、NEI 加上偏差，直接最大化 dev 的 CFEVER Score（只有 2 個參數）-----
GRID = np.arange(-4, 4.01, 0.25)


def fit_bias(logits, y, ok):
    best = (-1.0, 0.0, 0.0)
    for b1 in GRID:
        for b2 in GRID:
            s = cf((logits + np.array([0.0, b1, b2])).argmax(1), y, ok) - 1e-4 * (abs(b1) + abs(b2))   # 同分時偏好較小的偏差
            if s > best[0]:
                best = (s, b1, b2)
    return np.array([0.0, best[1], best[2]])


perm = np.random.default_rng(SEED).permutation(len(dev_y))
halves = [perm[:len(perm) // 2], perm[len(perm) // 2:]]
cross = []     # 二折交叉驗證：用一半 dev 找偏差、在另一半驗證，估計「校準真正帶來的提升」
for a, b in ((0, 1), (1, 0)):
    bias = fit_bias(dev_logits[halves[a]], dev_y[halves[a]], dev_ev_ok[halves[a]])
    cross.append((cf((dev_logits[halves[b]] + bias).argmax(1), dev_y[halves[b]], dev_ev_ok[halves[b]]),
                  cf(dev_logits[halves[b]].argmax(1), dev_y[halves[b]], dev_ev_ok[halves[b]])))
cf_cross, cf_cross_base = float(np.mean([c[0] for c in cross])), float(np.mean([c[1] for c in cross]))
USE_CAL = cf_cross > cf_cross_base + 0.005          # 交叉驗證至少提升 0.5 個百分點才採用
CAL_BIAS = fit_bias(dev_logits, dev_y, dev_ev_ok) if USE_CAL else np.zeros(3)
print(f"校準：交叉驗證 CFEVER {cf_cross_base:.4f} -> {cf_cross:.4f}；採用校準 = {USE_CAL}；偏差(supports/refutes/NEI) = {CAL_BIAS}")
dev_pred = (dev_logits + CAL_BIAS).argmax(1)
assert abs(cfever_score(dev, dev_pred, EV_OUT["dev"]) - cf(dev_pred, dev_y, dev_ev_ok)) < 1e-9   # 與前面的 CFEVER 公式一致

# 上限分析：若檢索完美（S/R 直接給標準證據），分類器能到多少？
gold_lists = []
for qi in range(len(dev["claim"])):
    if dev["label"][qi] == 2:
        gold_lists.append(EV["dev"][qi])
    else:
        g = min(dev["gold"][qi], key=lambda g: (len({t for t, _ in g}), len(g)))
        items = []
        for t, s in g:
            j = title2idx.get(t)
            items += [(0.0, j, sid, text, ()) for sid, text, _ in (get_sentences(j) if j is not None else ()) if sid == s]
        all_gold = {(t, s) for gg in dev["gold"][qi] for t, s in gg}
        fill = [c for c in EV["dev"][qi] if (titles[c[1]], c[2]) not in all_gold][:max(0, CLS_CTX_K - len(items))]   # 補滿 4 句，與訓練時的輸入長度一致
        gold_lists.append(items + fill)
oracle_logits = classify(dict(claim=dev["claim"], name="dev-oracle"), gold_lists)
acc_oracle = float((oracle_logits.argmax(1) == dev_y).mean())

metrics = dict(
    page_recall_at5=page_recall(dev, ranked["dev"], 5), page_recall_at_topk=page_recall(dev, ranked["dev"], PAGE_TOPK),
    evidence_recall_at5=evidence_recall(dev, EV_OUT["dev"], 5), evidence_recall_at5_reranker=evidence_recall(dev, EV["dev"], 5), use_evidence_selector=bool(SEL_ON), selector_feature_set=SEL_SET, selector_oof_recall=SEL_OOF, use_reranker2=bool(USE_RR2), evidence_recall_at5_no_hop=evidence_recall(dev, rr["dev"][0], 5), hop_delta=HOP_DELTA,
    classification_acc_no_calibration=acc_plain, classification_acc_old_prior_correction=acc_prior,
    classification_acc_calibrated=float((dev_pred == dev_y).mean()), classification_acc_oracle_evidence=acc_oracle,
    active_members=ACTIVE_MEMBERS, member_dev_acc=member_acc, ensemble_subsets=subset_df.round(4).to_dict("records"),
    cfever_score_no_calibration=cf_plain, cfever_score_calibration_crossfit=cf_cross, cfever_score_crossfit_baseline=cf_cross_base,
    use_calibration=bool(USE_CAL), calibration_bias=CAL_BIAS.tolist(), cfever_score=cf(dev_pred, dev_y, dev_ev_ok),
)
per_label = {LABEL_OUT[k]: float((dev_pred[dev_y == k] == k).mean()) for k in range(3)}
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("各類別 recall（最終預測）：", {k: round(v, 4) for k, v in per_label.items()})
print("混淆矩陣（列=真實，欄=預測；supports/refutes/NEI），未校準：")
print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_logits.argmax(1), name="pred")))
if USE_CAL:
    print("校準後：")
    print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_pred, name="pred")))


_rows9, _prev = [], _v7d
for name, tags in STEPS:
    lg = ens9(tags, "dev")
    r = {"步驟": name, "成員數": len(tags), "dev acc": round(float((lg.argmax(1) == dev_y).mean()), 4)}
    for s_, (d, n10, n01, z) in zip(DEV_SHARES, paired_at_shares(_prev, lg)):
        r[f"比上一步 @NEI{s_}"] = f"{d:+.4f}（+{n10}/-{n01}, z {z:+.1f}）"
    for s_, (d, n10, n01, z) in zip(DEV_SHARES, paired_at_shares(_v7d, lg)):
        r[f"比 v7 @NEI{s_}"] = f"{d:+.4f}（z {z:+.1f}）"
    _rows9.append(r)
    _prev = lg
print("逐步加入新成員（dev CFEVER；每一格是在相同 NEI 預測比例下的配對比較）")
print(pd.DataFrame(_rows9).to_string(index=False))


acc9 = float((dev_logits9.argmax(1) == dev_y).mean())
print(f"\nv9 最終集成（{len(V9_TAGS)} 成員）：dev accuracy = {acc9:.4f}（v7 為 {(_v7d.argmax(1) == dev_y).mean():.4f}）")
print("混淆矩陣（列=真實，欄=預測；supports/refutes/NEI），未加偏差：")
print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_logits9.argmax(1), name="pred")))

各成員 dev accuracy（未校準）： {'macbert_base': 0.7137, 'roberta_base': 0.7227, 'erlangshen_110m': 0.747, 'erlangshen_330m': 0.7663, 'erlangshen_330m_b': 0.7647, 'qwen06_pilot': 0.7103} | 集成： 0.778
成員組合的 dev 表現（未校準）：
                                                                                 members    acc  cfever
             macbert_base+roberta_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b 0.7790  0.7397
                          macbert_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b 0.7813  0.7397
                                            roberta_base+erlangshen_110m+erlangshen_330m 0.7783  0.7383
             macbert_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b+qwen06_pilot 0.7803  0.7377
macbert_base+roberta_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b+qwen06_pilot 0.7780  0.7370
                                          roberta_base+erlangshen_110m+erlangshen_330m_b 0.7757  0.7363
                                            macbert_base+erlang


加入 erlangshen_330m_b 的配對比較（dev CFEVER）：0.7330 → 0.7370（差 +0.0040）
  變好 34 題、變壞 22 題 → 標準誤約 0.0025；差值 / 標準誤 = +1.60
  |差值| < 2 倍標準誤 = 統計上分不出高下；此時留著新成員也無妨（平均通常不會變差），但別只憑這個數字換掉已確認分數的提交檔。
校準：交叉驗證 CFEVER 0.7370 -> 0.7463；採用校準 = True；偏差(supports/refutes/NEI) = [0.  0.5 1. ]



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2449.12it/s]


分類 dev-oracle / macbert_base:   0%|          | 0/47 [00:00<?, ?it/s]


分類 dev-oracle / macbert_base:   2%|▏         | 1/47 [00:11<08:59, 11.72s/it]

... (138 outputs left out)


分類 dev-oracle / qwen06_pilot:  98%|█████████▊| 184/188 [02:05<00:03,  1.16it/s]


分類 dev-oracle / qwen06_pilot: 100%|██████████| 188/188 [02:09<00:00,  1.46it/s]

{
  "page_recall_at5": 0.9175,
  "page_recall_at_topk": 0.9245,
  "evidence_recall_at5": 0.878,
  "evidence_recall_at5_reranker": 0.8615,
  "use_evidence_selector": true,
  "selector_feature_set": "base",
  "selector_oof_recall": {
    "base": {
      "1": 0.497,
      "3": 0.8275,
      "5": 0.878
    },
    "base+link": {
      "1": 0.501,
      "3": 0.8285,
      "5": 0.878
    },
    "base+link+rr2": {
      "1": 0.5105,
      "3": 0.8305,
      "5": 0.878
    }
  },
  "use_reranker2": true,
  "evidence_recall_at5_no_hop": 0.8615,
  "hop_delta": null,
  "classification_acc_no_calibration": 0.778,
  "classification_acc_old_prior_correction": 0.7833333333333333,
  "classification_acc_calibrated": 0.7806666666666666,
  "classification_acc_oracle_evidence": 0.803,
  "active_members": [
    "macbert_base",
    "roberta_base",
    "erlangshen_110m",
    "erlangshen_330m",
    "erlangshen_330m_b",
    "qwen06_pilot"
  ],
  "member_dev_acc": {
    "macbert_base": 0.7136666666666667,
    "r

逐步加入新成員（dev CFEVER；每一格是在相同 NEI 預測比例下的配對比較）
                步驟  成員數  dev acc            比上一步 @NEI0.42            比上一步 @NEI0.46             比上一步 @NEI0.5   比 v7 @NEI0.42   比 v7 @NEI0.46    比 v7 @NEI0.5
          v7（6 成員）    6   0.7780   +0.0000（+0/-0, z +0.0）   +0.0000（+0/-0, z +0.0）   +0.0000（+0/-0, z +0.0） +0.0000（z +0.0） +0.0000（z +0.0） +0.0000（z +0.0）
pilot → 完整版 qwen06    6   0.7840 +0.0063（+34/-15, z +2.7） +0.0053（+33/-17, z +2.3） +0.0033（+34/-24, z +1.3） +0.0063（z +2.7） +0.0053（z +2.3） +0.0033（z +1.3）
    + xlmr_xnli_k6    7   0.7907 +0.0010（+29/-26, z +0.4） +0.0030（+30/-21, z +1.3） +0.0030（+31/-22, z +1.2） +0.0073（z +2.6） +0.0083（z +3.1） +0.0063（z +2.1）
    + e330_sib2_k4    8   0.7933 +0.0063（+38/-19, z +2.5） +0.0010（+29/-26, z +0.4） +0.0067（+40/-20, z +2.6） +0.0137（z +4.3） +0.0093（z +3.1） +0.0130（z +3.8）
    + xlmr_sib2_k4    9   0.7960 +0.0017（+20/-15, z +0.8） +0.0043（+31/-18, z +1.9） +0.0027（+22/-14, z +1.3） +0.0153（z +4.3） +0.0137（z +3.9） +0.0157（z +4.3）
    + e330_sib3_k6   

## 5. Validation：分開觀察三階段的表現

Document Recall、Evidence Recall 都要求完整涵蓋任一組標準證據，且只統計可驗證的題目。
Label Accuracy 只看分類；CFEVER Score 還要求證據正確，NEI 則只看標籤。
三個階段已各自印出分數；下面彙整，並額外計算競賽使用的 CFEVER Score。
公開測試集沒有標準答案，因此分數在 dev.jsonl 上計算，test.jsonl 只輸出預測。
所有分數的範圍都是 0～1，例如 0.8 表示 80%。



In [18]:
# 部署用的類別偏差（選擇依據見 Report Q3、Q4(c)）。
# (1) v9：讓 test 的 NEI 預測比例對齊 v7（6 成員）在偏差 1.25～2.25 時的比例 —— 這是照 public 調的；
#     private 快照（0.8346）證實它沒有轉移：public 的 A 組 NEI 題特別多，private 的 A 組幾乎都是 supports / refutes。
# (2) v10 / v11 / v12：只用 dev 決定。dev 與 CFEVER test（= A 組）一樣三類各 1000 題（CFEVER 論文 Table 3），所以 dev 是 A 組的無偏估計。
#     v10 只調 NEI 偏差；v11 把 refutes 與 NEI 偏差一起調（supports 固定 0）；v12 用新證據與重新推論的集成，兩種規則取交叉驗證分數較高者。
# 最終提交兩份：v12 偏差 0（不加偏差）與 v12 的交叉驗證偏差；private 取兩者較高者。
V9_REF_SHARE = 1.75            # v9 public 最高的那份（11/23 前的最終提交），只作對照
V9_BIAS = {}
for _vb in (1.25, 1.5, 1.75, 1.875, 2.0, 2.25):
    _share = nei_share(test_logits, _vb)
    V9_BIAS[_vb] = (_share, bias_for_share(test_logits9, _share))


def dev_hits_v(bv):   # 每題是否得分（與 Kaggle metric 一致：標籤要對，S/R 的前 5 句證據要完整涵蓋任一組標準證據）
    _p = (dev_logits9 + np.asarray(bv)).argmax(1)
    return (_p == np.array(dev_y)) & ((np.array(dev_y) == 2) | np.array(dev_ev_ok, dtype=bool))


def dev_hits(b):      # 只加 NEI 偏差
    return dev_hits_v([0, 0, b])


def cv_select(grid, hits):   # 5 折：依 id 連續切塊（兄弟 claim 不會跨折）；在 4 折上選、在剩下 1 折上評估
    _res = []
    for _f in np.array_split(_order, 5):
        _tr = np.setdiff1d(_order, _f)
        _k = int(hits[:, _tr].mean(1).argmax())
        _res.append((grid[_k], hits[_k, _f].mean()))
    return _res


_order = np.argsort(np.array(dev["id"]))

# v10：只調 NEI 偏差
BIAS_GRID = np.round(np.arange(-1.0, 2.01, 0.25), 2)
_H = np.array([dev_hits(b) for b in BIAS_GRID])
_cv = cv_select(list(BIAS_GRID), _H)
CV_BIAS = float(_cv[0][0]) if len({b for b, _ in _cv}) == 1 else float(np.median([b for b, _ in _cv]))
print("dev CFEVER 對 NEI 偏差：", {float(b): round(float(h.mean()), 4) for b, h in zip(BIAS_GRID, _H)})
print("5 折交叉驗證選出的偏差：", [float(b) for b, _ in _cv], "→ CV CFEVER", round(float(np.mean([s for _, s in _cv])), 4))
_d = dev_hits(CV_BIAS).astype(int) - dev_hits(0.0).astype(int)
print(f"偏差 {CV_BIAS} vs 0 的配對比較：+{(_d > 0).sum()} / -{(_d < 0).sum()}，z = {_d.sum() / np.sqrt((_d != 0).sum()):.2f}（不顯著 → v10 用不加偏差的 0）")
DEV_BIASES = (0.0, CV_BIAS)      # 0 = 不加偏差（最終提交之一）；CV_BIAS = dev 5 折交叉驗證的最佳 NEI 偏差（對照用）

# v11：refutes 與 NEI 偏差一起調（網格 -1.0～3.0，間隔 0.25）
_g2 = np.round(np.arange(-1.0, 3.01, 0.25), 2)
BIAS_GRID2 = [(float(_r), float(_n)) for _r in _g2 for _n in _g2]
_H2 = np.array([dev_hits_v([0, _r, _n]) for _r, _n in BIAS_GRID2])
_cv2 = cv_select(BIAS_GRID2, _H2)
_r11, _n11 = collections.Counter(b for b, _ in _cv2).most_common(1)[0][0]
V11_BIAS = (0.0, _r11, _n11)     # (supports, refutes, NEI)
print("v11：5 折交叉驗證選出的 (refutes, NEI) 偏差：", [b for b, _ in _cv2], "→ CV CFEVER", round(float(np.mean([s for _, s in _cv2])), 4))
_d2 = dev_hits_v(V11_BIAS).astype(int) - dev_hits(0.0).astype(int)
print(f"v11 偏差 {V11_BIAS} vs 不加偏差的配對比較：+{(_d2 > 0).sum()} / -{(_d2 < 0).sum()}，z = {_d2.sum() / np.sqrt((_d2 != 0).sum()):.2f}")

# v12：新證據（EV12）＋本機成員重新推論後的集成；偏差同樣只用 dev 的 5 折決定，兩種規則（只調 NEI / refutes 與 NEI 一起調）取 CV 分數較高者
dev_ev_ok12 = np.array([lab == 2 or any(all(x in {(titles[c[1]], c[2]) for c in EV12["dev"][qi][:5]} for x in g) for g in dev["gold"][qi])
                        for qi, lab in enumerate(dev_y)])


def dev_hits12(bv):
    _p = (dev_logits12 + np.asarray(bv)).argmax(1)
    return (_p == np.array(dev_y)) & ((np.array(dev_y) == 2) | dev_ev_ok12)


_cv12a = cv_select([(0.0, 0.0, float(b)) for b in BIAS_GRID], np.array([dev_hits12([0, 0, b]) for b in BIAS_GRID]))
_cv12b = cv_select([(0.0, _r, _n) for _r, _n in BIAS_GRID2], np.array([dev_hits12([0, _r, _n]) for _r, _n in BIAS_GRID2]))
_s12a, _s12b = float(np.mean([s for _, s in _cv12a])), float(np.mean([s for _, s in _cv12b]))
print("v12：只調 NEI 的 5 折選擇：", [b[2] for b, _ in _cv12a], f"→ CV CFEVER {_s12a:.4f}")
print("v12：refutes 與 NEI 一起調的 5 折選擇：", [b[1:] for b, _ in _cv12b], f"→ CV CFEVER {_s12b:.4f}")
V12_BIAS = collections.Counter(b for b, _ in (_cv12a if _s12a >= _s12b else _cv12b)).most_common(1)[0][0]
V12_NAME = f"nei{V12_BIAS[2]}" if V12_BIAS[1] == 0 else f"r{V12_BIAS[1]}_nei{V12_BIAS[2]}"
for _nm, _bv in (("不加偏差", (0, 0, 0)), (V12_NAME, V12_BIAS)):
    _d = dev_hits12(_bv).astype(int) - dev_hits(0.0).astype(int)
    print(f"v12 {_nm} vs v10（原本證據、不加偏差）的配對比較：+{(_d > 0).sum()} / -{(_d < 0).sum()}，z = {_d.sum() / np.sqrt((_d != 0).sum()):.2f}")

strict_hits = {f"v9_share{V9_REF_SHARE}": dev_hits(V9_BIAS[V9_REF_SHARE][1]).mean()}
strict_hits.update({f"v10_nei{_b}": dev_hits(_b).mean() for _b in DEV_BIASES})
strict_hits[f"v11_r{_r11}_nei{_n11}"] = dev_hits_v(V11_BIAS).mean()
strict_hits["v12_nei0.0"], strict_hits[f"v12_{V12_NAME}"] = dev_hits12((0, 0, 0)).mean(), dev_hits12(V12_BIAS).mean()

metrics9 = {
    "document_recall_at5": page_recall(dev, ranked["dev"], 5),
    "evidence_recall_at5": evidence_recall(dev, EV_OUT["dev"], 5),
    "evidence_recall_at5_v12": evidence_recall(dev, EV12["dev"], 5),
    "classification_accuracy_v12": float((dev_logits12.argmax(1) == np.array(dev_y)).mean()),
    "classification_accuracy": acc9,
    **{f"cfever_score_{k}": v for k, v in strict_hits.items()},
}
print(f"Document Retrieval Recall@5: {metrics9['document_recall_at5']:.4f}")
print(f"Evidence Retrieval Recall@5: {metrics9['evidence_recall_at5']:.4f}（v12：{metrics9['evidence_recall_at5_v12']:.4f}）")
print(f"Classification Accuracy（12 成員集成，未加偏差）: {metrics9['classification_accuracy']:.4f}（v12：{metrics9['classification_accuracy_v12']:.4f}）")
print(f"CFEVER Score（dev，v9 部署偏差 {V9_BIAS[V9_REF_SHARE][1]:.3f}，對齊 v7 偏差 {V9_REF_SHARE}）: {strict_hits[f'v9_share{V9_REF_SHARE}']:.4f}")
for _b in DEV_BIASES:
    print(f"CFEVER Score（dev，v10 NEI 偏差 {_b}）: {strict_hits[f'v10_nei{_b}']:.4f}")
print(f"CFEVER Score（dev，v11 refutes 偏差 {_r11}、NEI 偏差 {_n11}）: {strict_hits[f'v11_r{_r11}_nei{_n11}']:.4f}")
print(f"CFEVER Score（dev，v12 不加偏差）: {strict_hits['v12_nei0.0']:.4f}")
print(f"CFEVER Score（dev，v12 偏差 {V12_BIAS}）: {strict_hits[f'v12_{V12_NAME}']:.4f}")

dev CFEVER 對 NEI 偏差： {-1.0: 0.735, -0.75: 0.7417, -0.5: 0.7477, -0.25: 0.7523, 0.0: 0.755, 0.25: 0.7557, 0.5: 0.7597, 0.75: 0.7583, 1.0: 0.757, 1.25: 0.752, 1.5: 0.7533, 1.75: 0.7413, 2.0: 0.7333}
5 折交叉驗證選出的偏差： [0.5, 0.5, 0.5, 0.5, 0.5] → CV CFEVER 0.7597
偏差 0.5 vs 0 的配對比較：+56 / -42，z = 1.41（不顯著 → v10 用不加偏差的 0）


v11：5 折交叉驗證選出的 (refutes, NEI) 偏差： [(1.0, 1.5), (1.0, 1.5), (1.0, 1.5), (1.0, 1.5), (1.0, 1.5)] → CV CFEVER 0.7637
v11 偏差 (0.0, 1.0, 1.5) vs 不加偏差的配對比較：+91 / -65，z = 2.08
v12：只調 NEI 的 5 折選擇： [1.0, 0.5, 0.5, 0.5, 0.5] → CV CFEVER 0.7650
v12：refutes 與 NEI 一起調的 5 折選擇： [(0.75, 1.25), (1.25, 1.75), (1.25, 1.75), (0.5, 0.5), (1.25, 1.25)] → CV CFEVER 0.7637
v12 不加偏差 vs v10（原本證據、不加偏差）的配對比較：+67 / -42，z = 2.39
v12 nei0.5 vs v10（原本證據、不加偏差）的配對比較：+93 / -54，z = 3.22
Document Retrieval Recall@5: 0.9175
Evidence Retrieval Recall@5: 0.8780（v12：0.8990）
Classification Accuracy（12 成員集成，未加偏差）: 0.7963（v12：0.8037）
CFEVER Score（dev，v9 部署偏差 1.981，對齊 v7 偏差 1.75）: 0.7337
CFEVER Score（dev，v10 NEI 偏差 0.0）: 0.7550
CFEVER Score（dev，v10 NEI 偏差 0.5）: 0.7597
CFEVER Score（dev，v11 refutes 偏差 1.0、NEI 偏差 1.5）: 0.7637
CFEVER Score（dev，v12 不加偏差）: 0.7633
CFEVER Score（dev，v12 偏差 (0.0, 0.0, 0.5)）: 0.7680


## 6. 產生 submission.csv

提交檔需要 `id`、`predicted_label`、`predicted_evidence` 三欄。
證據使用 `json.dumps()` 轉成合法 JSON 字串，NEI 的證據填 `[]`。
保持測試資料的 ID 與順序，再從 Kaggle Notebook 輸出提交這份 CSV。



In [19]:
def make_submission(pred, path, ev=None):
    ev = EV_OUT["test"] if ev is None else ev                       # 送出的證據：預設為原本的選擇器結果；v12 用 EV12
    evs = ["[]" if p == 2 else json.dumps([[titles[c[1]], c[2]] for c in ev[qi][:5]], ensure_ascii=False) for qi, p in enumerate(pred)]
    out = pd.DataFrame({"id": test["id"], "predicted_label": [LABEL_OUT[p] for p in pred], "predicted_evidence": evs})
    out.to_csv(path, index=False, encoding="utf-8")
    return out


_sfx = "_quick" if QUICK_TEST else ""
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_g9 = np.where(np.array(test["id"]) <= 31996, "A", np.where(np.array(["最喜歡" in c for c in test["claim"]]), "B", "C"))
# test 內有 3 種來源明顯不同的題目（見 Report Q4(d)）：A = id<=31996（與 train/dev 的 id 交錯）；
# B = id>31996 且含「最喜歡」（1000 題，虛構的個人偏好句）；C = id>31996 其餘（2000 題，整句像是抄自維基）。
V9_SUBS = {}
for _vb, (_share, _b9) in V9_BIAS.items():
    _pred = (test_logits9 + np.array([0, 0, _b9])).argmax(1)
    _path = OUTPUT_DIR / f"submission_v9_m{len(V9_TAGS)}_share{_vb}{_sfx}.csv"
    make_submission(_pred, _path)                                 # 證據使用選擇器重排後的前 5 句（EV_OUT）
    V9_SUBS[_vb] = _path
    print(f"  {_path.name}：test NEI 比例 {_share:.3f}（= v7 偏差 {_vb}）→ v9 偏差 {_b9:.3f}；各組 NEI 比例",
          {g: round(float((_pred[_g9 == g] == 2).mean()), 3) for g in "ABC"})

V10_SUBS = {}
for _b in DEV_BIASES:
    _pred = (test_logits9 + np.array([0, 0, _b])).argmax(1)
    _path = OUTPUT_DIR / f"submission_v10_m{len(V9_TAGS)}_nei{_b}{_sfx}.csv"
    make_submission(_pred, _path)
    V10_SUBS[_b] = _path
    print(f"  {_path.name}：NEI 偏差 {_b}；各組預測 (S, R, NEI)",
          {g: np.bincount(_pred[_g9 == g], minlength=3).tolist() for g in "ABC"})

_pred = (test_logits9 + np.array(V11_BIAS)).argmax(1)
V11_SUB = OUTPUT_DIR / f"submission_v11_m{len(V9_TAGS)}_r{V11_BIAS[1]}_nei{V11_BIAS[2]}{_sfx}.csv"
make_submission(_pred, V11_SUB)
print(f"  {V11_SUB.name}：refutes 偏差 {V11_BIAS[1]}、NEI 偏差 {V11_BIAS[2]}；各組預測 (S, R, NEI)",
      {g: np.bincount(_pred[_g9 == g], minlength=3).tolist() for g in "ABC"})

V12_SUBS = {}
for _nm, _bv in (("nei0.0", (0.0, 0.0, 0.0)), (V12_NAME, V12_BIAS)):
    _pred = (test_logits12 + np.array(_bv)).argmax(1)
    _path = OUTPUT_DIR / f"submission_v12_m{len(V9_TAGS)}_{_nm}{_sfx}.csv"
    make_submission(_pred, _path, ev=EV12["test"])
    V12_SUBS[_nm] = _path
    print(f"★ 最終提交 {_path.name}：偏差 {_bv}；證據 v12；各組預測 (S, R, NEI)", {g: np.bincount(_pred[_g9 == g], minlength=3).tolist() for g in "ABC"})

submission = pd.read_csv(V12_SUBS["nei0.0"])
print("最終提交：", V12_SUBS["nei0.0"].name, "與", V12_SUBS[V12_NAME].name)
print("輸出筆數：", len(submission))
print("輸出位置：", V12_SUBS["nei0.0"].resolve())
submission.head()

  submission_v9_m12_share1.25.csv：test NEI 比例 0.397（= v7 偏差 1.25）→ v9 偏差 1.337；各組 NEI 比例 {'A': 0.453, 'B': 1.0, 'C': 0.011}
  submission_v9_m12_share1.5.csv：test NEI 比例 0.414（= v7 偏差 1.5）→ v9 偏差 1.631；各組 NEI 比例 {'A': 0.485, 'B': 1.0, 'C': 0.013}
  submission_v9_m12_share1.75.csv：test NEI 比例 0.433（= v7 偏差 1.75）→ v9 偏差 1.981；各組 NEI 比例 {'A': 0.523, 'B': 1.0, 'C': 0.015}
  submission_v9_m12_share1.875.csv：test NEI 比例 0.442（= v7 偏差 1.875）→ v9 偏差 2.135；各組 NEI 比例 {'A': 0.539, 'B': 1.0, 'C': 0.017}
  submission_v9_m12_share2.0.csv：test NEI 比例 0.451（= v7 偏差 2.0）→ v9 偏差 2.269；各組 NEI 比例 {'A': 0.556, 'B': 1.0, 'C': 0.018}


  submission_v9_m12_share2.25.csv：test NEI 比例 0.467（= v7 偏差 2.25）→ v9 偏差 2.513；各組 NEI 比例 {'A': 0.584, 'B': 1.0, 'C': 0.024}
  submission_v10_m12_nei0.0.csv：NEI 偏差 0.0；各組預測 (S, R, NEI) {'A': [1046, 927, 1027], 'B': [0, 0, 1000], 'C': [1002, 982, 16]}
  submission_v10_m12_nei0.5.csv：NEI 偏差 0.5；各組預測 (S, R, NEI) {'A': [1014, 860, 1126], 'B': [0, 0, 1000], 'C': [1001, 982, 17]}
  submission_v11_m12_r1.0_nei1.5.csv：refutes 偏差 1.0、NEI 偏差 1.5；各組預測 (S, R, NEI) {'A': [890, 900, 1210], 'B': [0, 0, 1000], 'C': [997, 986, 17]}


★ 最終提交 submission_v12_m12_nei0.0.csv：偏差 (0.0, 0.0, 0.0)；證據 v12；各組預測 (S, R, NEI) {'A': [1101, 955, 944], 'B': [0, 0, 1000], 'C': [1006, 992, 2]}
★ 最終提交 submission_v12_m12_nei0.5.csv：偏差 (0.0, 0.0, 0.5)；證據 v12；各組預測 (S, R, NEI) {'A': [1060, 881, 1059], 'B': [0, 0, 1000], 'C': [1006, 991, 3]}
最終提交： submission_v12_m12_nei0.0.csv 與 submission_v12_m12_nei0.5.csv
輸出筆數： 6000
輸出位置： /home/yiting/NLP-Assignment3/output/final/submission_v12_m12_nei0.0.csv


,id,predicted_label,predicted_evidence
0,29579,refutes,"[[""沈懋孝"", 1], [""沈懋孝"", 0], [""明朝"", 4], [""明朝"", 24]..."
1,3895,supports,"[[""鄱陽湖"", 0], [""鄱陽"", 3], [""江西省"", 1], [""鄱陽湖"", 2]..."
2,24522,refutes,"[[""赫伯特·胡佛"", 1], [""胡佛"", 5], [""赫伯特·胡佛"", 0], [""胡佛..."
3,11060,NOT ENOUGH INFO,[]
4,23607,supports,"[[""馬肯森級大巡洋艦"", 7], [""馬肯森級大巡洋艦"", 0], [""馬肯森級大巡洋艦""..."


# Report

Your Kaggle username: 17568160 

Your public leaderboard score: 0.889

Your private leaderboard score:


1. Describe your method for retrieving Wikipedia documents (5%) and report Recall@N on development set (5%).

Answer: 

**Method: hybrid retrieval + a learned ranker, over the full Wikipedia dump (not the "index only the training pages" approach of the sample code).**
The sample code only indexes the pages used by the training set, but only 10.8% of the dev evidence pages are among them, so I index the whole wiki (1,187,751 pages) and keep only "title + first 110 characters" of each page as its retrieval summary. For every claim, three complementary cheap signals generate candidate pages, and a learned ranker re-ranks them:

- **Title matching**: wiki titles that appear as a substring of the claim (only titles of length 2–24; single-character titles appear in 13 claims on average and are almost all noise; about 80% of the dev evidence pages have a title that appears verbatim in the claim). At most 6 pages per title, at most 40 candidates per claim.
- **BM25**: character bigrams (hashing, 2^21 dimensions, k1 = 1.2, b = 0.75); top 50, of which the top 20 become candidates. This handles claims that do not literally contain the title.
- **Dense retrieval**: `BAAI/bge-small-zh-v1.5` (about 24M parameters) encodes the same summaries; top 50, of which the top 20 become candidates. This handles paraphrases and synonyms.
- **Learned ranker**: the union of the three candidate sets is scored by a HistGradientBoosting classifier (19 features: title length and position, whether the title is contained in a longer title, BM25 / dense scores and ranks, which signals hit, ...) that predicts "is this page a gold evidence page". It is trained on CFEVER train only, and the training-set scores are 3-fold cross-fitted predictions so that the score distribution at training time matches inference time. The top 6 pages go to the next stage.

**Recall@N (development set).** The denominator is the 2,000 supports / refutes dev claims that have gold evidence. A claim counts as a hit if the top-N pages cover all pages of at least one *complete* gold evidence group.

| Recall@N | title match only | BM25 only | dense only | RRF fusion (no learning) | **learned ranker (this method)** |
|---|---|---|---|---|---|
| @1 | 0.5235 | 0.5400 | 0.5235 | 0.6995 | **0.7915** |
| @3 | 0.7405 | 0.6385 | 0.6575 | 0.8290 | **0.8955** |
| @5 | 0.8255 | 0.6725 | 0.6965 | 0.8535 | **0.9175** |
| @6 (actually used) | 0.8385 | 0.6845 | 0.7135 | 0.8570 | **0.9245** |
| @10 | 0.8635 | 0.7175 | 0.7555 | 0.8760 | **0.9375** |
| @20 | 0.9000 | 0.7585 | 0.7975 | 0.9220 | **0.9410** |

Each signal alone reaches only 0.67–0.83 at @5, plain RRF fusion reaches 0.8535, and the learned ranker raises it to 0.9175 (+6.4 points). What is passed on to the next stage is the top 6 pages (Recall@6 = 0.9245).

**Models used in this question and how.** Only the dense retriever uses a pre-trained model, `BAAI/bge-small-zh-v1.5` (about 24M parameters, public Hugging Face checkpoint), **used as is, without fine-tuning**: every page's "title + first 110 characters" and every claim is encoded into a vector ([CLS] embedding, L2-normalised) and the top 50 are taken by inner product. Title matching and BM25 are rule-based / statistical methods; the learned ranker is scikit-learn's HistGradientBoosting trained from scratch on CFEVER train only. (Using the sibling claims' page scores as extra ranker features was also tried and not adopted, see Q4(e).)

**For the whole pipeline:** every model has fewer than 1B parameters, no external inference API is called, and all training uses only the CFEVER train / dev data and the provided Wikipedia dump. The pre-trained checkpoints used for evidence retrieval and classification are described in the "Models used" paragraphs of Q2 and Q3.

2. Describe your method for retrieving evidence (5%) and report Recall@N on development set  (5%).

Answer: 

**Method: coarse filtering → cross-encoder re-ranking → evidence-sentence selector.**

- **Candidate sentences**: from the top 6 pages (only the first 60 sentences of each page, because 95% of the gold evidence is within the first 50 sentences), a character-bigram overlap score keeps 32 sentences per claim.
- **Sentence re-ranker (cross-encoder)**: `hfl/chinese-macbert-base` (about 102M) fine-tuned with input `claim [SEP] page title: sentence` (many sentences start with 「他」 or 「該」, so adding the title supplies the missing subject), max length 128, 1 epoch, lr 3e-5, batch 16 × 2 gradient accumulation. Positives are the gold evidence sentences; negatives are other sentences of the same page, sentences of other retrieved pages, and sentences of the pages hyperlinked from the gold evidence (the hardest negatives). 5 negatives per claim (the 3 highest-scoring hard negatives + 2 random ones).
- **Evidence-sentence selector**: classification accuracy is about 0.78 but the CFEVER score is only 0.73; the gap comes from "the top 5 sentences do not cover a complete gold evidence group" (the candidate pool actually covers about 96% of the claims). The re-ranker only sees "claim + one sentence" and cannot see the other sentences of the page, the page rank or the sentence position, so I train a HistGradientBoosting selector on the supports / refutes dev claims that re-ranks the top 12 sentences. Its target is "does this sentence belong to any gold evidence group", and its features are: re-ranker score, rank, gap to the top-1 score, sentence index, the page probability and page rank, the number of sentences of the same page in the top 5, character-bigram overlap with the claim, and the newly covered part of the claim. The selector only changes which 5 sentences are submitted as evidence; it does not change the classifier input.
- **Evaluation**: the selector's dev numbers are 5-fold (grouped by claim) out-of-fold predictions, so it never sees its own gold labels; for test, the selector trained on all of dev is used.
- **Submitted evidence**: claims predicted as NOT ENOUGH INFO get `[]`, all other claims get the top 5 sentences.
- **Final improvement (11/24): a larger candidate pool and a second, pre-trained re-ranker.** (i) The 60-sentences-per-page limit was losing gold evidence: on dev, a complete gold evidence group lies inside the 32-sentence pool for 90.45% of the supports / refutes claims, but inside the top 6 pages for 92.45%. The pool now takes *all* sentences of the top 6 pages and keeps the 96 best by the same bigram score (49 sentences per claim on average), which reaches the page-level bound of 0.9245. (ii) Every pool sentence is also scored by `BAAI/bge-reranker-v2-m3`, used as is. On its own it is weaker than the fine-tuned MacBERT (dev Recall@5 0.834 versus 0.8725 in the larger pool), but its errors are different, so the two scores are z-normalised (with dev statistics) and mixed as (1 − w)·MacBERT + w·bge. (iii) The same evidence selector then re-orders the top 12, with one of the two raw scores in the six feature slots that an earlier second-re-ranker experiment had used. w ∈ {0, 0.5, 0.75, 1} and which score feeds the selector are chosen by the 5-fold OOF Recall@5 on dev: w = 0.5, with the bge score.

**Recall@N (development set).** Same denominator (2,000 supports / refutes claims); a claim counts as a hit if the top-N sentences cover at least one complete gold evidence group. (For multi-sentence evidence the whole group must be in the top N, so @1 cannot be high.)

| Recall@N | sentence re-ranker (MacBERT, 32-sentence pool) | + evidence selector (32-sentence pool, 5-fold OOF) | **final: 96-sentence pool + MacBERT / bge fusion + selector (5-fold OOF)** |
|---|---|---|---|
| @1 | 0.4955 | 0.4970 | **0.5080** |
| @3 | 0.8005 | 0.8275 | **0.8475** |
| @5 (actually submitted) | 0.8615 | 0.8780 | **0.8990** |

The selector raises Recall@5 from 0.8615 to 0.8780. By evidence size, claims needing 1 / 2 / 3 sentences go from 0.9356 → 0.9442, 0.7856 → 0.8129 and 0.6509 → 0.6864, so multi-sentence claims benefit most. The final larger pool + fused re-ranker raises Recall@5 further to 0.899 (+2.1 points); the larger pool alone (MacBERT + selector) gives 0.8905. For reference, the classification upper bound with *perfect* evidence is a dev accuracy of 0.803, versus 0.778 with our retrieved evidence.

**Models used in this question and how.** The sentence re-ranker is `hfl/chinese-macbert-base` (about 102M, public Hugging Face checkpoint) with a 2-class head, **fine-tuned on CFEVER train only** (settings above). The evidence selector is scikit-learn's HistGradientBoosting trained only on the CFEVER dev supports / refutes claims (evaluated by 5-fold OOF). Earlier I also tried `hfl/chinese-roberta-wwm-ext` (about 102M, fine-tuned on CFEVER train only, with hard-negative mining) as a second re-ranker; it gave no gain and is not used in the final pipeline. The final pipeline adds `BAAI/bge-reranker-v2-m3` (about 568M parameters < 1B; a public cross-encoder checkpoint that BAAI had already trained on external retrieval data), **used as is, without fine-tuning** (inference only, fp16, max length 192), to score the candidate sentences; such already-trained checkpoints are allowed by the TA (see the last paragraph of Q3). Fine-tuning it on CFEVER train was also tried and not adopted (Q4(e)).

3. Describe your classification method (5%) and report Accuracy on development set  (5%).

Answer: 

**Method: an ensemble of 12 classifiers (each < 1B parameters). Input: claim + the top 4 evidence sentences; the members' log-softmax outputs are averaged with equal weights and a bias is then added to the NOT-ENOUGH-INFO logit.**

- **Input**: `[CLS] claim [SEP] title: evidence1 [SEP] title: evidence2 …`, where the evidence is the top 4 sentences of the re-ranker (97% of the gold evidence groups have at most 3 sentences). NLI-pre-trained models get the evidence first (the NLI "premise, hypothesis" corresponds to "evidence, claim"), and their classification head is re-ordered from entailment / contradiction / neutral to supports / refutes / NEI.
- **The key to the training data**: the training inputs of all three classes must look like the test-time inputs. For all three classes the evidence is built from "what the same re-ranker actually returns for the training claims": take the top 4 sentences (their order is shuffled with 30% probability); for supports / refutes, if a gold evidence sentence is not among the top 4, the missing gold sentence is inserted and the lowest-scoring non-gold sentence is dropped; for NOT ENOUGH INFO the re-ranker's top 4 are used as they are. (The first version made a mistake here, see Q4(a).)
- **Sibling claims**: CFEVER claims are generated in groups — one source sentence is rewritten into several claims (supports / refutes / NEI mixed) with adjacent ids. Six members get 2–3 "sibling claims" appended after the evidence: within the same data split, claims whose id differs by ≤ 20 and whose character-bigram Jaccard similarity is ≥ 0.1, taking the most similar ones (only claim text and id are used, **no labels**). The model can then compare "what this sentence changed relative to its siblings".
- **The 12 members and their dev accuracy (single model, no bias):**

| member | parameters | trained | dev accuracy |
|---|---|---|---|
| `hfl/chinese-macbert-base` | ~102M | local | 0.7137 |
| `hfl/chinese-roberta-wwm-ext` | ~102M | local | 0.7227 |
| `IDEA-CCNL/Erlangshen-Roberta-110M-NLI` | ~110M | local | 0.7470 |
| `IDEA-CCNL/Erlangshen-Roberta-330M-NLI` (seed 22 / 42; first 8 layers frozen, Adafactor) | ~326M | local | 0.7663 / 0.7647 |
| `Qwen/Qwen3-0.6B-Base` + LoRA (all training data, 2 epochs) | ~0.6B | local | 0.7630 |
| `Qwen/Qwen3-0.6B-Base` + LoRA + sibling claims | ~0.6B | local | **0.7753** |
| `joeddav/xlm-roberta-large-xnli` (full fine-tuning, 6 evidence sentences, trained on train + half of dev, cross-fit) | ~560M | Kaggle GPU | 0.7707 (OOF) |
| `Erlangshen-Roberta-330M-NLI` + 2 sibling claims (full fine-tuning) | ~326M | Kaggle GPU | 0.7697 |
| `Erlangshen-Roberta-330M-NLI` + 3 sibling claims, 6 evidence sentences (full fine-tuning) | ~326M | Kaggle GPU | 0.7720 |
| `xlm-roberta-large-xnli` + 2 sibling claims | ~560M | Kaggle GPU | 0.7657 |
| `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` + 2 sibling claims | ~278M | Kaggle GPU | 0.7553 |
| **12-member ensemble** | | | **0.7963** |

**The classifiers re-read the final evidence.** Changing only the submitted evidence (Q2) raises dev CFEVER by just 0.33 points (0.7550 → 0.7583), because the classifier had also given a wrong label (usually NEI) to most of the claims whose evidence used to be missing. So the 7 locally trained members (weights unchanged, inference only) now read the top 4 sentences of the final evidence; the 5 Kaggle-trained members keep their original input, because only their logits were downloaded, not their weights. Dev accuracy of the local members: MacBERT 0.7137 → 0.7203, RoBERTa 0.7227 → 0.7210, Erlangshen-110M 0.7470 → 0.7580, Erlangshen-330M 0.7663 → 0.7667 / 0.7647 → 0.7600, Qwen3 0.7630 → 0.7720, Qwen3 + siblings 0.7753 → 0.7820; **12-member ensemble 0.7963 → 0.8037**.

("Kaggle GPU" = trained on a Kaggle Notebook 16GB GPU with the same code, see the `KG_SPECS` / `KG_SCRIPT` definitions in the Classification → Training code cell of this notebook; my local RTX 3050 has only 4GB. OOF = dev is split into two contiguous halves by id and each half is predicted by a model that has not seen it.)

**Models used in this question and how.** All 12 members in the table are public Hugging Face checkpoints and are **fine-tuned on CFEVER data only** (train; the members marked "cross-fit" additionally use half of dev); no external dataset is used. Training: the encoder-type members (MacBERT, RoBERTa, Erlangshen, XLM-R, mDeBERTa) get a 3-class head and are fully fine-tuned (the local 330M model freezes the embeddings and the first 8 layers and uses Adafactor because of GPU memory; the others use AdamW, lr 8e-6 – 2e-5, 2 epochs). Erlangshen, XLM-R and mDeBERTa are checkpoints that others have already fine-tuned on public NLI data, so their NLI head is kept and re-ordered to supports / refutes / NEI. `Qwen/Qwen3-0.6B-Base` uses frozen bf16 weights + LoRA (r = 16, all projection layers) + a classification head, lr 1e-4, 2 epochs. At inference the 12 members' log-softmax outputs are averaged with equal weights.

**Checkpoints trained on external data, and the TA's permission.** Erlangshen NLI, XLM-R XNLI and mDeBERTa XNLI above (and `BAAI/bge-reranker-v2-m3` in Q2) are checkpoints that others had already trained on external NLI / retrieval data. I confirmed with the TA that such checkpoints are allowed (email reply of 2026-11-14: "that is fine, as long as the parameter count is at most 1B"; the TA's reply on the discussion board: "any open pre-trained model / checkpoint may be used, but in each question of the report you must state where you used which model and how it was used / trained", which is why Q1–Q4 each have a "Models used" paragraph). No model above 1B was used; only the model weights were downloaded, and the datasets these checkpoints were originally trained on were neither downloaded nor used.

**Accuracy (development set, 3,000 claims, 1,000 per class; ensemble, no bias) = 0.8037 for the final ensemble** (the same 12 members with the old evidence: 0.7963; an earlier 6-member ensemble: 0.7780); the corresponding dev CFEVER score = 0.7633 (old evidence: 0.7550; 6 members: 0.7370). **Class biases and the two final submissions** (details in Q4(c)): my first choice, `submission_v9_m12_share1.75.csv`, added an NEI bias of 1.981 chosen with reference to the public leaderboard (52.3% of group A predicted NEI; public 0.889). The private snapshot of 11/23 showed that it did not generalise (private 0.8346), so the biases are now chosen on dev only. The two final submissions are (1) `submission_v12_m12_nei0.0.csv`, **no bias** (dev CFEVER 0.7633; group A predicted as 1101 / 955 / 944 supports / refutes / NEI, close to the 1000 / 1000 / 1000 of the CFEVER test set; public 0.862), and (2) `submission_v10_m12_nei0.0.csv`, the same no-bias rule on the 12-member ensemble with the old evidence (32-sentence pool) (dev CFEVER 0.7550; public 0.862), kept as a guard in case the new evidence pipeline fails on test in a way dev does not show. `submission_v12_m12_nei0.5.csv` (NEI +0.5 chosen by 5-fold CV on dev; dev CFEVER 0.7680; public 0.872) was also submitted but not selected. On 11/23 the final pair was the old-evidence ensemble with no bias (public 0.862) and with refutes +1.0 / NEI +1.5 (`submission_v11_m12_r1.0_nei1.5.csv`, public 0.877).

4. … Anything that can strengthen your report. (10%)

Answer: 

**(a) A shortcut in the training-data construction (first version).** In the first version the NOT ENOUGH INFO evidence always had 4 sentences, while supports / refutes was "gold evidence + 0–3 random distractors", so about half of them had fewer than 4 sentences and the gold evidence was always ranked first. The model learned "4 sentences = NEI", and on dev 43% of the supports / refutes claims were predicted NEI; public was only 0.716. After building the evidence of all three classes from the re-ranker's actual output, public became 0.833.

**(b) Composition of the test set.** The 6,000 test claims fall into three groups (my own observation of the data, not official information): group A (id ≤ 31996, 3,000 claims) is of the same type as train / dev; group B (id > 31996 and containing 「最喜歡」 "favourite", 1,000 claims) are made-up personal-preference sentences that the model predicts as NEI 100% of the time; group C (the remaining 2,000 claims) are sentences copied directly from a Wikipedia page (≈ supports) or with just one number changed (≈ refutes), and the model's prediction agrees with "was only a number changed" in about 99% of them. So almost all differences in score come from group A.

**(c) The class biases: what went wrong on private, and the fix.**

*First choice.* Dev has 1/3 of each class and its best NEI bias is 0.5 (5-fold CV), but public preferred a much larger bias (own submissions: the earlier 6-member ensemble at bias 1.0 / 1.5 / 2.0 → 0.871 / 0.881 / 0.876; the 12-member ensemble with the NEI share aligned to the 6-member bias 1.5 / 1.75 / 1.875 / 2.0 → 0.883 / **0.889** / 0.888 / 0.888). Label-shift estimation (BBSE) said that group A is almost balanced (about 0.33 / 0.30 / 0.37) and a domain classifier could not separate dev from group A (AUC 0.55), so I concluded that group A differs in its *conditional* distribution and chose the bias from public (52.3% / 55.6% of group A predicted NEI).

*What the private snapshot showed.* On 11/23 the private snapshot was **0.8346** (the better of the two selected files; public 0.889). The cause is not the model but the public / private split:
- Group A is the original CFEVER test set (3,000 claims, ids interleaved with train / dev), which has exactly 1,000 claims per class (CFEVER paper, Table 3) — the same distribution as dev. So dev is an unbiased estimate of group A, and the BBSE estimate above was right.
- The total over public + private matches dev: for share 1.75 (the file the 0.8346 belongs to according to the fit below), public + private = 2,667 + 2,504 = 5,171 correct claims out of 6,000; dev predicts 2,201 for a balanced group A at this bias (dev CFEVER 0.7337) plus about 2,970 for groups B / C (≈ 99%), i.e. 5,171. The model performed exactly as dev predicted; only the *split* of the rows differs.
- If the split were random, share 1.75 should score about 0.862 on both halves, and removing the bias should *raise* public (to ≈ 0.873). Instead public was 0.889 and private 0.8346, and removing the bias *lowered* public (0.862, see below).
- Fitting all 9 of my own public scores with dev's per-class hit rates (a grid over how many supports / refutes / NEI claims of group A and how many group B / C claims fall into public): a random split fits about 100× worse than the best fits, and every good fit puts **750–1,000 of group A's 1,000 NEI claims into public**. Private group A is therefore almost all supports / refutes, where a large NEI bias is expensive: it turns refutes into NEI (dev hit rate on refutes: 0.722 at bias 0, 0.531 at bias 1.981). Using only the 9 public scores (not the private snapshot), the fit predicts private **0.834** for share 1.75 — the observed value is 0.8346.
- A falsifiable check: a fit that does not use the scores of the two dev-only submissions predicts public 0.859 / 0.871 for bias 0 / 0.5; the uploads scored 0.862 / 0.872.

*The fix: choose the bias on dev only.* Since dev and group A have the same class distribution, the bias is chosen on dev alone: 5-fold CV (contiguous id blocks) picks 0.5 in every fold (CV CFEVER 0.7597), but 0.5 versus 0 is not significant (+56 / −42 claims, z = 1.41), so the corrected submission uses **no bias at all** (dev CFEVER 0.7550). Its group-A prediction distribution (1046 / 927 / 1027) is also the closest to the published 1000 / 1000 / 1000. I deliberately did *not* use a negative bias, although the fit predicts a higher private score for it: that would be tuning to the inferred private composition (the same mistake in the other direction) and it lies outside the range that dev supports (dev CFEVER drops to 0.735 at bias −1).

*The refutes and NEI biases together, also on dev only.* Earlier a refutes bias had been rejected because public dropped (0.881 → 0.875); in hindsight public was not a fair judge for a class-prior parameter, because its group A is NEI-heavy. Redone on dev only for the 12-member ensemble: a grid of refutes and NEI biases (each from −1.0 to 3.0 in steps of 0.25, supports fixed at 0) with the same 5 folds picks **refutes +1.0, NEI +1.5 in every fold** (an interior point of the grid); CV CFEVER 0.7637 versus 0.7550 with no bias, paired +91 / −65 claims (z = 2.08). Group A is predicted as 890 / 900 / 1210, and public is 0.877 (the composition fit had predicted 0.875).

*Final pair of 11/23.* No bias (public 0.862) and refutes +1.0 / NEI +1.5 (public 0.877). Each rests on a different piece of evidence: the joint bias is what the dev-only rule selects, and no bias is what the composition analysis favours (the fit, now including the joint-bias score, expects private ≈ 0.884 for no bias and ≈ 0.877 for the joint bias), so the pair also covers the case where that analysis is off. The first choice (share 1.75) was dropped from the final pair because its private score is already known to be at most 0.8346.

*Better evidence, the same rules.* After the final evidence improvement (Q2, Q3) the same dev-only rules were applied to the new ensemble. The NEI-only rule picks +0.5 in 4 of the 5 folds (CV CFEVER 0.7650); the joint refutes / NEI rule now picks four different pairs in the five folds (CV 0.7637), so the rule with the higher CV score, NEI only, is used. Paired against the old evidence with no bias: the final ensemble without bias +67 / −42 claims (z = 2.39), with NEI +0.5 +93 / −54 (z = 3.22).

*Final submissions.* The final ensemble without bias (`submission_v12_m12_nei0.0.csv`, public 0.862) and the old-evidence ensemble without bias (`submission_v10_m12_nei0.0.csv`, public 0.862); the private leaderboard uses the better of the two. The composition fit expects private ≈ 0.891 for the final ensemble without bias, ≈ 0.882 for it with NEI +0.5, ≈ 0.884 for the old evidence without bias and ≈ 0.877 for the old evidence with the joint bias; it does not use any score of the final ensemble. Against the old evidence, the final ensemble without bias wins +58 / −17 on the dev supports / refutes claims (z = 4.73), which make up most of private. I kept the old-evidence file rather than the final ensemble with NEI +0.5 as the second file: it guards against a failure of the new evidence pipeline on test (a model risk), at the cost of having no NEI-biased fallback if private holds more NEI claims than the fit estimates. The public scores of the final ensemble equal those of the corresponding old-evidence files (0.862 / 0.872 for no bias / NEI +0.5), and the fit had predicted 0.862 for the unbiased file before the upload: public holds most of group A's NEI claims, whose score does not depend on evidence, while the evidence gain is on supports / refutes claims, which are mostly in private.

**(d) Adding members step by step: every step is tested with a paired test at the same dev NEI-prediction share (0.42 / 0.46 / 0.50).**

| step | dev acc | vs. 6 members (three shares; paired-test z) | public |
|---|---|---|---|
| earlier 6-member ensemble | 0.7780 | — | 0.881 |
| pilot → full Qwen3-0.6B | 0.7840 | +0.63 / +0.53 / +0.33 pt (z 2.7 / 2.3 / 1.3) | |
| + XLM-R-large-XNLI, + 330M with sibling claims | 0.7933 | +1.37 / +0.93 / +1.30 pt (z 4.3 / 3.1 / 3.8) | |
| + XLM-R with siblings, + 330M with siblings (3) | 0.7953 | +1.53 / +1.33 / +1.73 pt (z 4.2 / 3.5 / 4.4) | 0.886 |
| + mDeBERTa with siblings | 0.7937 | +1.73 / +1.43 / +1.80 pt (z 4.5 / 3.6 / 4.5) | 0.884 |
| + Qwen3-0.6B with siblings (final, 12 members) | 0.7963 | +1.67 / +1.73 / +2.30 pt (z 4.2 / 4.2 / 5.6) | 0.883 (**0.889** when the NEI share is aligned to 1.75) |

(The public column, except for the last row, uses the share aligned to the 6-member bias 1.5.) All three larger ensembles are above the 6-member one on public; the differences among them (0.883–0.886) change only a few dozen predictions and are within the public noise; the dev paired tests decide the member set, so I use all 12 members. Leave-one-out shows that the largest contributions come from the three new families / new inputs (removing the 330M with siblings, XLM-R or the full Qwen costs 0.3–0.7 pt each), whereas just changing the seed of the same model barely helps (a second full Qwen agrees with the first on 91% of the claims and costs −0.1 pt when added; a fully fine-tuned 330M on Kaggle agrees with the two existing 330M models on 90% and changes ±0). In other words, an ensemble needs members that *make different errors*: different model families or different input information (sibling claims).

**Models mentioned in this question and how they were used.** All models used in these experiments are already listed in Q1–Q3 (bge-small-zh used as is; MacBERT / RoBERTa / Erlangshen / XLM-R-XNLI / mDeBERTa-XNLI / Qwen3-0.6B all fine-tuned on CFEVER only). The members tried in (d) but not adopted — a second Qwen3-0.6B + LoRA (only the seed differs), Erlangshen-330M-NLI fully fine-tuned with 6 evidence sentences, and the cross-fit versions of XLM-R / Qwen — were trained as in Q3; the cluster-aware selector and the sibling-claim stacking in (e) are scikit-learn HistGradientBoosting / logistic regression trained on CFEVER dev only.

**(e) Directions that were tried but not adopted (all compared on the same dev claims with paired tests; without enough evidence they were not adopted).**

(The code of the second hop and of the second re-ranker is still in the Evidence Retrieval code cell and its result can be seen in that cell's output; the code of the other experiments is not in the final notebook, to keep it separate from the final pipeline, and the numbers below are the outputs from when the experiments were run.)

- Second hop (adding pages via the hyperlinks of the top-2 sentences of the first round): evidence Recall@5 0.8615 → 0.8635, which did not pass the threshold set in advance.
- A second re-ranker (hard-negative mining) as a selector feature: OOF Recall@5 stayed at 0.8780.
- Cluster-aware evidence selector (adding the sibling claims' re-ranking results as features): 5-fold OOF Recall@5 (folds contiguous by id) 0.875 → 0.879, but public 0.877 → 0.878; the difference is too small, not adopted.
- Cluster-aware page ranking (sibling claims' page scores as features): dev page Recall@6 0.9245 → 0.9295, but it changes the top-1 page of 2–6% of group C claims (which are almost all correct), so the risk outweighs the gain; not adopted.
- Averaging two evidence views (each member reads the re-ranker's top 4 and the selector's top 4 and the results are averaged): dev accuracy 0.778 → 0.784, but the gain only appears at NEI bias 0; at the deployed NEI share the paired results are +27/−24, +21/−22, +26/−26, and public 0.881 → 0.877; not adopted.
- Sibling-claim stacking (a logistic regression trained on dev, with the ensemble probabilities + the siblings' predictions and text differences as input): dev OOF +0.4–0.8 pt, but it is a post-hoc adjustment fitted on dev, the same kind as the refutes bias (works on dev, not necessarily on test), so the evidence is insufficient; not adopted. When the same sibling-claim information was instead *put into the model input and trained on train*, dev and public improved together.
- Fine-tuning `BAAI/bge-reranker-v2-m3` on CFEVER train (Kaggle GPU; hard negatives mined with the un-tuned model, listwise loss, 1 epoch): on its own it became much stronger (Recall@5 in the larger pool 0.834 → 0.8835), but after the fusion with MacBERT and the selector the 5-fold OOF Recall@5 was 0.8985 versus 0.8990 for the un-tuned model, so the un-tuned model is kept.
- Learning the ensemble weights, a refutes bias and a dev calibration on dev: (earlier experiments; the code is not in the final notebook) they worked on dev but public got worse or unstable. (Public was later shown to be a biased judge for class-prior parameters, see (c); the refutes bias was later redone on dev only.)

**(f) Error analysis (final ensemble, dev NEI-prediction share 0.46).** For the supports claims whose gold evidence reached the classifier's top 4 sentences (834 claims), the FEVER accuracy is 0.893; for refutes (857 claims) it is only 0.662, and 229 of them are predicted NEI — "seeing contradicting evidence but answering not enough information" is the largest remaining error source, and it is exactly what the large NEI bias of the first choice made worse on the refutes-heavy private set (see (c)). There are 309 supports / refutes claims whose evidence did not reach the classifier (page not found: 151, page found but sentence not in the top 4: 158); these claims can hardly score, and this is the room left on the retrieval side. The final evidence improvement (Q2) works on the second part: the larger pool and the fused re-ranker raise evidence Recall@5 from 0.878 to 0.899.

**(g) Lessons and limitations.**
- Post-hoc adjustments fitted on dev (stacking, learned ensemble weights, two-view averaging) were not reliable; the only things that improved dev and public together are "stronger and more diverse base models". Class-prior parameters (the NEI / refutes bias, calibration) were at first judged on public, which turned out to be the wrong judge (see the third point).
- Evidence and label are coupled: better evidence alone moved dev CFEVER by only 0.33 points, because the claims whose evidence had been missing were mostly mislabelled too; the gain (0.83 points, z = 2.39) appeared only after the classifiers re-read the new evidence.
- Every change was judged with a paired test, and public differences were converted into noise units (when a submission changes only a few dozen predictions, ±0.003 is noise); a change was adopted only when both sides agreed.
- The NEI share chosen with reference to public did not generalise to private, because public is not a random sample of the test classes. When the dataset paper tells us that dev and test have the same class distribution, class-prior parameters should be chosen on dev only; a public preference that contradicts dev is a property of the public subset, not of the test set. The gain from the ensemble itself does not depend on this and is verified on dev with paired tests (z 4–5.6).

5. Generative AI Usage Statement

Answer: 

I used Anthropic's Claude (through Claude Code; Claude Sonnet 5.5 and Claude Opus 5.5) as a supporting tool throughout this assignment, mainly for code implementation, debugging, experimental design discussions, statistical analysis of the results, and report writing. The first code cell of this notebook also starts with a short Generative AI usage note that points to this question.

My own contributions included setting the goal and the rules of the work: the score should be improved only by methods whose gain can be verified on the development set, every model must stay under 1B parameters, no external inference API may be used, and training may use only the CFEVER data. Before using checkpoints that others had already fine-tuned on external NLI / retrieval data, I asked the TA by email and on the discussion board whether they were allowed, and I made sure that each question of this Report states which models it uses and how. Because my local GPU has only 4GB, I decided to run the training of the larger classifiers on the Kaggle Notebook GPU of my own account.

In the early stages of the project, I proposed most of the directions for improving the model: choosing different training data from the provided data, correcting problems in the training data, trying different pre-trained models, and strengthening the models that worked. I also discussed optimisation plans with Claude, weighed the options it suggested against my own, and decided which ones to try next.

During the optimisation process, I compared the development-set results and the public scores of each modification, judged whether a change was a real improvement or only noise, and decided which approaches should be kept or dropped (Q4(d), (e)). After the private snapshot of 11/23 showed that the NEI bias chosen with reference to the public leaderboard did not generalise, I decided to stop tuning class-prior parameters on public and to cite only my own account's submissions in this Report. I then chose to spend the remaining time on evidence retrieval and set a deadline for that work, after which no further changes would be made. For the final submissions, I chose to pair the final ensemble with the old-evidence file, as a guard against a failure of the new evidence pipeline on test, rather than with the NEI-biased variant that the analysis had suggested (Q4(c)).

Claude assisted in translating these ideas into executable code (retrieval, re-rankers, evidence selector, classifiers and ensemble, the Kaggle GPU training scripts, submission generation), troubleshooting implementation issues, designing the paired tests and the public / private composition analysis, interpreting the results, and improving the clarity and organisation of the Report. All code was run by me on my own computer (Windows + WSL2, RTX 3050 4GB) and on my own Kaggle account. I examined the outputs, checked the numbers in this Report against them, submitted the files to Kaggle, selected the final two submissions, and revised the layout and wording of the Report myself.